# Notebook 04 — D200 TAIL paired replications (seeds43 and44)

Run All on a fresh Colab T4 GPU. This notebook automatically runs seed43, then seed44.
Complete both RGB replications here before running Notebook05.

Seed42 is historical and is never trained or resumed by this queue. Only a verified checkpoint of the exact current seed/run may resume. Completed runs with verified checkpoints and evidence are skipped. An interrupted seed pauses the queue; rerunning resumes its durable state before advancing.

The scientific engine and configuration are unchanged from the completed D200 TAIL seed42 experiment, except the training seed. The separately identified wrapper creates a fresh Python namespace for each run; disposable preflight is followed by a seeded production rebuild. No smoke-test weights enter training.

Fixed budget: 4,500 successful optimizer updates per seed; cosine horizon10,000; warmup500; validation/milestones every500; effective batch64 (16x4); denoising early stopping OFF. Both best and final checkpoints remain available. No training, policy rollout, or final evaluation is claimed before actual execution.


## 1. Locked queue and scientific provenance

Default MODE="train" runs/resumes the two additional seeds. MODE="evaluate" permits reports only for existing runs; it cannot create training. Do not run concurrent sessions for the same Run ID. Hyperparameter changes require a separately named experiment, not this replication queue.


In [ ]:
MODALITY='rgb'
TRAINING_SEEDS=[43,44]
MODE="train"
ENGINE_META={'code_version': 'notebook04-05-paired-gpu-d200-tail-v1', 'shared_build_sha256': '62c010b9cf90328a4534614c1fb1ad5ab914d61bd48b5ce429084c99460d7aba', 'source_snapshot_sha256': 'f904181d8fe71c67bff17b42a20a5e226ec045bdd1bf1feaa099c569329e8a65', 'dataset_id': 'd200_paired_tail_v1_88bac96f5257', 'data_fingerprint': '7e86659ebc890c6ce6aa7ec6907b3bfa63d0045ed30a580aac475b89384e15c7', 'normalization_hash': '8cfb1b1d216f5c0c9752628ef745fa601c7ee1667c445e5c0d7b5a6dfc1355a5', 'run_ids': {'rgb': 'd200_rgb_paired_gpu_tail_fixed4500_v1_seed42_f4221f88a689', 'events': 'd200_events_paired_gpu_tail_fixed4500_v1_seed42_78e67b2b54a1'}, 'module_hashes': {'04_diffusion.py': '97a85cda954c910fa14c33c1572197d5cc4a8502e824abcfe224644ca45f7e54', '08_evidence.py': 'c2b5b2efc33cdc2f6ae8da6899c8b767b902d680c3a38a1ce8e1c6e5d3a8b990', '06_checkpoints.py': 'b09919c50fe1fae80ee0fa6c9ba579a1fa62940e8c3f118dc59f185b0d14366d', '12_summary.py': '69e97ba70479235c01537a35d44b0dfdfa0be9ea7d70bca55f009f2b60fc851d', '10_diagnostics.py': 'cabfb7d8e4a39c07f6e1e92528330fe4cd32dabb4627ba12debfcd3756d99110', '07_preflight.py': '2bff2dc16ea78fe44786f90bcf0817c15bc30af1be68f084e22030052cd75f23', '05_validation.py': '72bd3774b1fd1b3058c5bfa121fe4badcb3465a649846a99661c8dbadaac84a2', '01_setup.py': 'ba4c98f8d810be5d77382696ee497002c292ef2ee19d38f091162c9048df1eb6', '11_sampling.py': '3f0a96baced967cae428c3d83263c51a882e697e898428e9350edf90ecb8e373', '09_train.py': '4510c9b8533fc6e04f86d179eafdf644f274fd66e5944e1fed0329f99f7ca02e', '02_data.py': '690155613c35d6f13bc1e43f9a6265eb64456b29e43b55d2016140b32a9787cc', '03_model.py': '0a5b176a9ee40aacd361988e09478d42b711c0efca3cef0a77e32bfb10fce06c'}}
RECIPES={'43': {'modality': 'rgb', 'visual_normalization': 'fixed_imagenet', 'event_std_floor': 0.001, 'event_channels': ['positive', 'negative', 'time_surface'], 'event_window_ms': 50.0, 'time_surface_tau_s': 0.05, 'experiment': 'paired_gpu_d200_tail_fixed4500_v1', 'seed': 43, 'split_seed': 20260927, 'policy_lr': 5e-05, 'encoder_lr': 5e-06, 'weight_decay': 1e-06, 'adam_betas': [0.95, 0.999], 'adam_eps': 1e-08, 'micro_batch': 16, 'accumulation': 4, 'eval_batch': 32, 'max_steps': 4500, 'warmup_steps': 500, 'lr_schedule_steps': 10000, 'min_lr_ratio': 0.1, 'validation_every': 500, 'evidence_every': 100, 'early_stop_enabled': False, 'early_stop_patience': 6, 'early_stop_min_step': 3000, 'early_stop_min_delta': 0.001, 'diffusion_steps': 100, 'max_beta': 0.999, 'clip_predicted_x0': False, 'unet_dims': [128, 256, 512], 'time_embed_dim': 128, 'unet_kernel': 5, 'group_norm_groups': 8, 'crop_size': 76, 'prediction_horizon': 16, 'observation_horizon': 2, 'execution_horizon': 8, 'action_dim': 7, 'state_per_observation': 18, 'state_contract': 'robot_and_gripper_qpos9_qvel9_without_cube', 'tail_target_policy': {'first_input': 2, 'last_input': 'T-1', 'observation_horizon': 2, 'prediction_horizon': 16, 'execution_horizon': 8, 'action_alignment': 'actions[t:t+16]; execute indices0:8', 'tail_padding': 'repeat_last_recorded_action_within_demo', 'observation_padding': 'none', 'loss_policy': 'unmasked_epsilon_MSE_all_16_targets_including_endpoint_repeats', 'action_is_recorded_scope': 'provenance indicator; NOT a training loss mask', 'normalization_scope': 'TRAIN_ONLY_OVERLAPPING_WINDOWS_INCLUDING_ENDPOINT_REPEATS', 'statistic_estimator': 'same overlapping-window mean/std method as historical export', 'terminal_semantics': 'Endpoint continuation convention; repeated command is not a measured post-demo action or guaranteed zero-motion hold.'}, 'export_schema': 3, 'dataset_id': 'd200_paired_tail_v1_88bac96f5257', 'bn_policy': 'imagenet_running_statistics_frozen_all_layers', 'ema_max_decay': 0.9999, 'ema_power': 0.75, 'ema_inv_gamma': 1.0, 'amp': True, 'initial_amp_scale': 1024.0, 'grad_clip_norm': 1.0, 'max_consecutive_nonfinite': 8, 'max_total_nonfinite': 50, 'validation_draws': 3, 'validation_seed': 70301, 'diagnostic_train_per_demo': 4, 'profile_samples_per_val_demo': 2, 'diagnostic_seed': 90413, 'milestone_kind': 'ema_fp16_weights_only_not_resumable', 'checkpoint_schema': 6, 'replicate_tag': 'main', 'resume_generations_to_keep': 2, 'condition_probe_timesteps': [0, 10, 25, 50, 75, 99]}, '44': {'modality': 'rgb', 'visual_normalization': 'fixed_imagenet', 'event_std_floor': 0.001, 'event_channels': ['positive', 'negative', 'time_surface'], 'event_window_ms': 50.0, 'time_surface_tau_s': 0.05, 'experiment': 'paired_gpu_d200_tail_fixed4500_v1', 'seed': 44, 'split_seed': 20260927, 'policy_lr': 5e-05, 'encoder_lr': 5e-06, 'weight_decay': 1e-06, 'adam_betas': [0.95, 0.999], 'adam_eps': 1e-08, 'micro_batch': 16, 'accumulation': 4, 'eval_batch': 32, 'max_steps': 4500, 'warmup_steps': 500, 'lr_schedule_steps': 10000, 'min_lr_ratio': 0.1, 'validation_every': 500, 'evidence_every': 100, 'early_stop_enabled': False, 'early_stop_patience': 6, 'early_stop_min_step': 3000, 'early_stop_min_delta': 0.001, 'diffusion_steps': 100, 'max_beta': 0.999, 'clip_predicted_x0': False, 'unet_dims': [128, 256, 512], 'time_embed_dim': 128, 'unet_kernel': 5, 'group_norm_groups': 8, 'crop_size': 76, 'prediction_horizon': 16, 'observation_horizon': 2, 'execution_horizon': 8, 'action_dim': 7, 'state_per_observation': 18, 'state_contract': 'robot_and_gripper_qpos9_qvel9_without_cube', 'tail_target_policy': {'first_input': 2, 'last_input': 'T-1', 'observation_horizon': 2, 'prediction_horizon': 16, 'execution_horizon': 8, 'action_alignment': 'actions[t:t+16]; execute indices0:8', 'tail_padding': 'repeat_last_recorded_action_within_demo', 'observation_padding': 'none', 'loss_policy': 'unmasked_epsilon_MSE_all_16_targets_including_endpoint_repeats', 'action_is_recorded_scope': 'provenance indicator; NOT a training loss mask', 'normalization_scope': 'TRAIN_ONLY_OVERLAPPING_WINDOWS_INCLUDING_ENDPOINT_REPEATS', 'statistic_estimator': 'same overlapping-window mean/std method as historical export', 'terminal_semantics': 'Endpoint continuation convention; repeated command is not a measured post-demo action or guaranteed zero-motion hold.'}, 'export_schema': 3, 'dataset_id': 'd200_paired_tail_v1_88bac96f5257', 'bn_policy': 'imagenet_running_statistics_frozen_all_layers', 'ema_max_decay': 0.9999, 'ema_power': 0.75, 'ema_inv_gamma': 1.0, 'amp': True, 'initial_amp_scale': 1024.0, 'grad_clip_norm': 1.0, 'max_consecutive_nonfinite': 8, 'max_total_nonfinite': 50, 'validation_draws': 3, 'validation_seed': 70301, 'diagnostic_train_per_demo': 4, 'profile_samples_per_val_demo': 2, 'diagnostic_seed': 90413, 'milestone_kind': 'ema_fp16_weights_only_not_resumable', 'checkpoint_schema': 6, 'replicate_tag': 'main', 'resume_generations_to_keep': 2, 'condition_probe_timesteps': [0, 10, 25, 50, 75, 99]}}
ORCHESTRATOR_SHA256='aee3c95a3dbbdd2ad1224f4e65e5557839bb7b7217f1a8ef83c39c8f9f03368d'


## 2. Verified engine and isolated queue controller

This self-contained payload includes all twelve original source modules. Their individual hashes and shared build are checked before execution. The outer controller applies only the selected training seed after setup and before data/model construction. All preprocessing statistics remain pinned and training-only; no source export is regenerated.


In [ ]:
SOURCE_MODULES={'01_setup.py': "# Run All entry point. Uses the existing Colab torch/torchvision CUDA pair.\nimport os, sys, json, math, time, random, hashlib, copy, gc, csv, shutil\nimport importlib.util, subprocess, platform, contextlib, traceback, uuid\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nNOTEBOOK_START_PERF = time.perf_counter()\n\nfor module, package in [('numpy', 'numpy'), ('h5py', 'h5py'), ('matplotlib', 'matplotlib')]:\n    if importlib.util.find_spec(module) is None:\n        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])\ntry:\n    import torch\n    import torchvision\nexcept Exception as exc:\n    raise RuntimeError('Select a fresh Colab GPU runtime with its matching torch/torchvision pair.') from exc\nimport numpy as np\nimport h5py\nimport matplotlib.pyplot as plt\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, default_collate\nfrom torchvision.models import resnet18, ResNet18_Weights\nfrom google.colab import drive\n\nassert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then Run All.'\nDEVICE = torch.device('cuda')\ndrive.mount('/content/drive')\nPROJECT_ROOT = Path('/content/drive/MyDrive/DiffusionPolicyEvents')\nRAW_DATASET = PROJECT_ROOT / 'data/raw/lift_ph_raw.hdf5'\nSOURCE_INDEX = PROJECT_ROOT / 'configs/processed_sources/paired_gpu_d200_tail_training_ready.json'\nDEV_SPLIT_PATH = PROJECT_ROOT / 'configs/training/d200_rgb_event_train_val.json'\nCHECKPOINT_ROOT = PROJECT_ROOT / 'artifacts/checkpoints' / MODALITY\nfor path in [RAW_DATASET, SOURCE_INDEX, DEV_SPLIT_PATH]:\n    assert path.is_file(), f'Missing paired source input: {path}. Complete the corrected tail-export Notebook03 first; Notebook02 source stays unchanged.'\n\nCODE_VERSION = 'notebook04-05-paired-gpu-d200-tail-v1'\nCODE_SHA256 = '__BUILD_CODE_SHA256__'\n# ALL experiment choices live here. Editing any value creates a different RUN_ID.\nMODE = 'train'  # Run All trains/resumes only this new paired experiment; evaluate never trains.\nEVAL_OPTIONS = {'sampling_windows': 20, 'sampling_seeds': [90413, 90414]}\nCFG = {\n    'modality': MODALITY, 'visual_normalization': ('fixed_imagenet' if MODALITY == 'rgb' else 'per_camera_channel_train_pixel_zscore'),\n    'event_std_floor': 1e-3, 'event_channels': ['positive', 'negative', 'time_surface'],\n    'event_window_ms': 50.0, 'time_surface_tau_s': .05,\n    'experiment': 'paired_gpu_d200_tail_fixed4500_v1', 'seed': 42, 'split_seed': 20260927,\n    'policy_lr': 5e-5, 'encoder_lr': 5e-6, 'weight_decay': 1e-6,\n    'adam_betas': [0.95, 0.999], 'adam_eps': 1e-8,\n    'micro_batch': 16, 'accumulation': 4, 'eval_batch': 32, 'max_steps': 4500,\n    'warmup_steps': 500, 'lr_schedule_steps': 10000, 'min_lr_ratio': 0.1,\n    'validation_every': 500, 'evidence_every': 100,\n    'early_stop_enabled': False, 'early_stop_patience': 6, 'early_stop_min_step': 3000,\n    'early_stop_min_delta': 0.001,\n    'diffusion_steps': 100, 'max_beta': 0.999, 'clip_predicted_x0': False,\n    'unet_dims': [128, 256, 512], 'time_embed_dim': 128,\n    'unet_kernel': 5, 'group_norm_groups': 8, 'crop_size': 76,\n    'prediction_horizon': 16, 'observation_horizon': 2, 'execution_horizon': 8,\n    'action_dim': 7, 'state_per_observation': 18,\n    'state_contract': 'robot_and_gripper_qpos9_qvel9_without_cube',\n    'tail_target_policy': {'first_input': 2, 'last_input': 'T-1', 'observation_horizon': 2, 'prediction_horizon': 16, 'execution_horizon': 8, 'action_alignment': 'actions[t:t+16]; execute indices0:8', 'tail_padding': 'repeat_last_recorded_action_within_demo', 'observation_padding': 'none', 'loss_policy': 'unmasked_epsilon_MSE_all_16_targets_including_endpoint_repeats', 'action_is_recorded_scope': 'provenance indicator; NOT a training loss mask', 'normalization_scope': 'TRAIN_ONLY_OVERLAPPING_WINDOWS_INCLUDING_ENDPOINT_REPEATS', 'statistic_estimator': 'same overlapping-window mean/std method as historical export', 'terminal_semantics': 'Endpoint continuation convention; repeated command is not a measured post-demo action or guaranteed zero-motion hold.'},\n    'export_schema': 3, 'dataset_id': 'd200_paired_tail_v1_88bac96f5257',\n    'bn_policy': 'imagenet_running_statistics_frozen_all_layers',\n    'ema_max_decay': 0.9999, 'ema_power': 0.75, 'ema_inv_gamma': 1.0,\n    'amp': True, 'initial_amp_scale': 1024.0, 'grad_clip_norm': 1.0,\n    'max_consecutive_nonfinite': 8, 'max_total_nonfinite': 50,\n    'validation_draws': 3, 'validation_seed': 70301,\n    'diagnostic_train_per_demo': 4, 'profile_samples_per_val_demo': 2,\n    'diagnostic_seed': 90413,\n    'milestone_kind': 'ema_fp16_weights_only_not_resumable',\n    'checkpoint_schema': 6, 'replicate_tag': 'main',\n    'resume_generations_to_keep': 2,\n    'condition_probe_timesteps': [0, 10, 25, 50, 75, 99],\n}\nassert MODE in ['train', 'evaluate']\nassert CFG['event_std_floor'] > 0\nassert EVAL_OPTIONS['sampling_windows'] > 0 and len(EVAL_OPTIONS['sampling_seeds']) >= 2\nassert CFG['micro_batch'] * CFG['accumulation'] == 64\nassert 0 < CFG['warmup_steps'] < CFG['max_steps'] <= CFG['lr_schedule_steps']\nassert CFG['max_steps'] % CFG['validation_every'] == 0\nassert isinstance(CFG['early_stop_enabled'], bool)\nassert CFG['prediction_horizon'] == 16 and CFG['observation_horizon'] == 2\nassert CFG['validation_every'] % CFG['evidence_every'] == 0\nassert CFG['early_stop_patience'] > 0 and CFG['validation_draws'] > 0\nassert not CFG['clip_predicted_x0'], 'z-score actions require unclipped diffusion x0.'\nassert CFG['resume_generations_to_keep'] >= 2\nassert all(d % CFG['group_norm_groups'] == 0 for d in CFG['unet_dims'])\nassert CFG['seed'] in [42, 43, 44] and CFG['split_seed'] == 20260927\nEXPECTED_DATASET_ID = 'd200_paired_tail_v1_88bac96f5257'\nEXPECTED_PAIRED_FINGERPRINT = '7e86659ebc890c6ce6aa7ec6907b3bfa63d0045ed30a580aac475b89384e15c7'\nEXPECTED_NORMALIZATION_HASH = '8cfb1b1d216f5c0c9752628ef745fa601c7ee1667c445e5c0d7b5a6dfc1355a5'\nEXPECTED_SOURCE_PROTOCOL_HASH = '93f4f6bf54a9469be3ae4e41e2d98dda71850f41a4ca863e865f152a052e3ab3'\nEXPECTED_PREPARATION_BUILD = '0dba021949ead3a91c4c40bf9ffe20e79b067061ae7c202c5e3188d09c28ae2d'\nEXPECTED_TRAIN_DEMOS, EXPECTED_VAL_DEMOS = 160, 40\nEXPECTED_TRAIN_SAMPLES, EXPECTED_VAL_SAMPLES = 7460, 1806\nEXPECTED_RAW_SHA256 = '68f2a8cfe2502de1224f0708fb82e46da07abce108c63e721d72fc1618272f37'\nEXPECTED_EXPORT_PROTOCOL_HASH = '88bac96f525758fdee4b2f059324bc90cce81f1255af30352fb8d6ecea49820e'\nEXPECTED_EXPORT_BUILD = '91f30c3f7c12b3b7ba79a2b4667f8613167ef6c1d0e1a8bd55ea2107b9757302'\nEXPECTED_PARENT_DATA_FINGERPRINT = '79257cd4b409d4125f682ee8517006d65ca3ff54dae9803aae06b480e2e3623b'\nEXPECTED_PARENT_NORMALIZATION_HASH = '65a31662ca4501984c1f40d7226a70c5d5e5b9752718f91c2e1f27e6ca34d193'\nassert CFG['tail_target_policy']['loss_policy']=='unmasked_epsilon_MSE_all_16_targets_including_endpoint_repeats'\nassert MODALITY in ['rgb', 'events'] and CFG['dataset_id'] == EXPECTED_DATASET_ID\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\ndef json_default(value):\n    if isinstance(value, np.integer): return int(value)\n    if isinstance(value, np.floating): return float(value)\n    if isinstance(value, np.bool_): return bool(value)\n    if isinstance(value, np.ndarray): return value.tolist()\n    if isinstance(value, Path): return str(value)\n    raise TypeError(f'Unsupported evidence type: {type(value).__name__}')\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + '.tmp-' + uuid.uuid4().hex)\n    try:\n        tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False, default=json_default), encoding='utf-8')\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n\nRUNTIME = {\n    'python': platform.python_version(), 'torch': torch.__version__,\n    'torchvision': torchvision.__version__, 'numpy': np.__version__, 'h5py': h5py.__version__,\n    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),\n    'capability': list(torch.cuda.get_device_capability(0)),\n}\nprint('Runtime:', RUNTIME)\nprint('Paired NVIDIA D200:', MODALITY, '| fresh ImageNet / fresh policy | dataset:', EXPECTED_DATASET_ID)\nprint('Mode:', MODE)\nprint('Run All includes real CUDA preflight. Full Colab execution has not been verified before delivery.')\n", '02_data.py': '# Load only the reviewed schema3 tail export. No legacy dataset or checkpoint fallback.\nCAMERAS = [\'agentview\', \'robot0_eye_in_hand\']\nIMAGE_KEYS = [c+\'_\'+s for c in CAMERAS for s in [\'previous\',\'current\']]\nCOMMON_KEYS = [\'control_steps\',\'control_timestamps\',\'proprio_previous\',\'proprio_current\',\'action_targets\',\'action_is_recorded\',\'action_source_indices\']\nARRAY_KEYS = set(IMAGE_KEYS + COMMON_KEYS)\n\ndef safe_child(root, relative):\n    root = Path(root).resolve(); relative = Path(relative)\n    assert not relative.is_absolute() and \'..\' not in relative.parts, \'Unsafe artifact path.\'\n    result = (root/relative).resolve()\n    assert result.is_relative_to(root)\n    return result\n\ndef verify_robot_state_layout(model_xml):\n    import xml.etree.ElementTree as ET\n    root = ET.fromstring(model_xml); world = root.find(\'worldbody\'); assert world is not None\n    joints = [e for e in world.iter() if e.tag in [\'joint\',\'freejoint\']]\n    names = [f\'robot0_joint{i}\' for i in range(1,8)] + [\'gripper0_right_finger_joint1\',\'gripper0_right_finger_joint2\',\'cube_joint0\']\n    assert [e.get(\'name\') for e in joints] == names\n    assert [(\'free\' if e.tag==\'freejoint\' else e.get(\'type\',\'hinge\')) for e in joints] == [\'hinge\']*7+[\'slide\']*2+[\'free\']\n\ndef validate_event_array(array, n):\n    assert array.shape == (n,3,84,84) and array.dtype == np.float32\n    assert np.isfinite(array).all() and (array>=0).all() and (array[:,2]<=1+1e-6).all()\n    active = (array[:,0]+array[:,1])>0\n    assert np.array_equal(active,array[:,2]>0), \'Event channel support mismatch.\'\n    if active.any(): assert array[:,2][active].min() >= math.exp(-1)-1e-4\n\ndef verify_paired_identity(index, dataset, protocol, normalization):\n    assert index[\'status\']==dataset[\'status\']==\'COMPLETED\' and index[\'schema\']==dataset[\'schema\']==3\n    assert index[\'dataset_id\']==dataset[\'dataset_id\']==EXPECTED_DATASET_ID==CFG[\'dataset_id\']\n    assert digest_json(protocol)==dataset[\'export_protocol_hash\']==index[\'export_protocol_hash\']==EXPECTED_EXPORT_PROTOCOL_HASH\n    assert protocol[\'version\']==dataset[\'code_version\']==\'paired-gpu-tail-export-d200-v1\'\n    assert protocol[\'build_sha256\']==dataset[\'export_code_sha256\']==EXPECTED_EXPORT_BUILD\n    assert protocol[\'source_protocol_hash\']==dataset[\'source_protocol_sha256\']==EXPECTED_SOURCE_PROTOCOL_HASH\n    assert protocol[\'raw_sha256\']==EXPECTED_RAW_SHA256\n    assert protocol[\'parent_data_fingerprint\']==EXPECTED_PARENT_DATA_FINGERPRINT\n    assert protocol[\'parent_normalization_hash\']==EXPECTED_PARENT_NORMALIZATION_HASH\n    assert protocol[\'policy\']==dataset[\'tail_target_policy\']==index[\'tail_target_policy\']==CFG[\'tail_target_policy\']\n    assert dataset[\'parent_dataset_id\']==protocol[\'parent_dataset_id\']==\'d200_paired_gpu_v2_93f4f6bf54a9\'\n    split=dataset[\'split\'];training=split[\'train_demos\'];validation=split[\'validation_demos\']\n    assert len(training)==len(set(training))==EXPECTED_TRAIN_DEMOS\n    assert len(validation)==len(set(validation))==EXPECTED_VAL_DEMOS and set(training).isdisjoint(validation)\n    assert split[\'source_split\']==\'D200\' and split[\'source_split_seed\']==CFG[\'split_seed\']==20260927\n    assert len(split[\'common_validation_demos\'])==10 and set(split[\'common_validation_demos\'])<=set(validation)\n    assert set(split[\'historical_d50_train_demos\'])<=set(training)\n    assert split[\'common_validation_demos\']==dataset[\'common_validation_demos\']\n    assert normalization[\'schema\']==3 and normalization[\'train_demos\']==training\n    assert normalization[\'tail_target_policy\']==CFG[\'tail_target_policy\']\n    assert normalization[\'scope\']==\'TRAIN_ONLY_OVERLAPPING_WINDOWS_INCLUDING_ENDPOINT_REPEATS\'\n    assert set(normalization[\'train_export_manifest_sha256\'])==set(training)\n    assert digest_json(normalization)==index[\'normalization_hash\']==dataset[\'normalization_hash\']==EXPECTED_NORMALIZATION_HASH\n    assert normalization[\'proprio_sample_count\']==EXPECTED_TRAIN_SAMPLES\n    assert normalization[\'action_target_count\']==EXPECTED_TRAIN_SAMPLES*16\n    assert normalization[\'endpoint_repeat_action_target_count\']==EXPECTED_TRAIN_DEMOS*120\n    identity={\'export_protocol\':digest_json(protocol),\'export_schema\':3,\'parent_data_fingerprint\':EXPECTED_PARENT_DATA_FINGERPRINT,\n        \'demos\':[{k:v for k,v in r.items() if k not in [\'absolute_directory\',\'relative_directory\']} for r in dataset[\'demos\']],\n        \'normalization\':digest_json(normalization),\'split\':split}\n    assert identity==dataset[\'fingerprint_basis\']\n    assert digest_json(identity)==index[\'paired_data_fingerprint\']==dataset[\'paired_data_fingerprint\']==EXPECTED_PAIRED_FINGERPRINT\n    assert len(dataset[\'demos\'])==len({r[\'demo_id\'] for r in dataset[\'demos\']})==EXPECTED_TRAIN_DEMOS+EXPECTED_VAL_DEMOS\n    assert {r[\'demo_id\'] for r in dataset[\'demos\']}==set(training+validation)\n    assert dataset[\'train_samples\']==EXPECTED_TRAIN_SAMPLES and dataset[\'validation_samples\']==EXPECTED_VAL_SAMPLES\n    assert dataset[\'state_per_observation\']==18 and dataset[\'state_total\']==36\n    assert dataset[\'observation_horizon\']==2 and dataset[\'prediction_horizon\']==16 and dataset[\'action_dim\']==7\n    return split\n\n\ndef verify_arrays(arrays, companion, states, actions):\n    assert set(arrays)==set(companion)==ARRAY_KEYS\n    for key in COMMON_KEYS:np.testing.assert_array_equal(arrays[key],companion[key])\n    steps=arrays[\'control_steps\'];times=arrays[\'control_timestamps\'];n=len(steps);T=len(actions)\n    assert states.shape==(T,32) and actions.shape==(T,7) and T>=18\n    assert np.isfinite(states).all() and np.isfinite(actions).all() and (np.abs(actions)<=1+1e-6).all()\n    assert steps.dtype==np.int64 and steps.ndim==1 and times.shape==(n,)\n    np.testing.assert_array_equal(steps,np.arange(2,T,dtype=np.int64));assert n==T-2\n    np.testing.assert_allclose(times,steps/20.,atol=1e-6,rtol=0)\n    robot=np.c_[states[:,1:10],states[:,17:26]].astype(np.float32)\n    for key,offset in [(\'proprio_previous\',-1),(\'proprio_current\',0)]:\n        assert arrays[key].shape==(n,18) and arrays[key].dtype==np.float32\n        np.testing.assert_allclose(arrays[key],robot[steps+offset],atol=1e-6,rtol=1e-6)\n    requested=steps[:,None]+np.arange(16,dtype=np.int64)\n    assert arrays[\'action_is_recorded\'].dtype==np.bool_ and arrays[\'action_is_recorded\'].shape==(n,16)\n    assert arrays[\'action_source_indices\'].dtype==np.int64 and arrays[\'action_source_indices\'].shape==(n,16)\n    np.testing.assert_array_equal(arrays[\'action_is_recorded\'],requested<T)\n    np.testing.assert_array_equal(arrays[\'action_source_indices\'],np.minimum(requested,T-1))\n    assert int((~arrays[\'action_is_recorded\']).sum())==120\n    assert arrays[\'action_targets\'].shape==(n,16,7) and arrays[\'action_targets\'].dtype==np.float32\n    np.testing.assert_array_equal(arrays[\'action_targets\'],np.asarray(actions,np.float32)[np.minimum(requested,T-1)])\n    for key in IMAGE_KEYS:\n        if MODALITY==\'rgb\':assert arrays[key].shape==(n,84,84,3) and arrays[key].dtype==np.uint8\n        else:validate_event_array(arrays[key],n)\n    assert all(np.isfinite(arrays[k]).all() for k in COMMON_KEYS)\n    return n\n\n\nSOURCE_RECORD = json.loads(SOURCE_INDEX.read_text())\nDATASET_ROOT = safe_child(PROJECT_ROOT,SOURCE_RECORD[\'relative_root\'])\nDATASET_MANIFEST_PATH = DATASET_ROOT/\'paired_dataset.json\'\nassert sha256_file(DATASET_MANIFEST_PATH)==SOURCE_RECORD[\'dataset_manifest_sha256\']\nDATASET_RECORD = json.loads(DATASET_MANIFEST_PATH.read_text())\nSOURCE_PROTOCOL = json.loads((DATASET_ROOT/\'protocol.json\').read_text())\nNORMALIZATION_PATH = safe_child(DATASET_ROOT,DATASET_RECORD[\'normalization_file\'])\nassert sha256_file(NORMALIZATION_PATH)==DATASET_RECORD[\'normalization_sha256\']\nNORMALIZATION = json.loads(NORMALIZATION_PATH.read_text())\nsplit = verify_paired_identity(SOURCE_RECORD,DATASET_RECORD,SOURCE_PROTOCOL,NORMALIZATION)\nassert json.loads(DEV_SPLIT_PATH.read_text())==split\nassert sha256_file(RAW_DATASET)==EXPECTED_RAW_SHA256\ntrain_ids,val_ids=split[\'train_demos\'],split[\'validation_demos\']\nCOMMON_VAL_IDS=split[\'common_validation_demos\']\nPAIRED_SAMPLE_FINGERPRINT=EXPECTED_PAIRED_FINGERPRINT\nDATA_FINGERPRINT=EXPECTED_PAIRED_FINGERPRINT\nNORMALIZATION_HASH=digest_json(NORMALIZATION)\nCACHE,DATA_AUDIT,EXCLUSIONS={ },[],[]\nwith h5py.File(RAW_DATASET,\'r\') as raw:\n    for record in DATASET_RECORD[\'demos\']:\n        demo=record[\'demo_id\'];folder=safe_child(DATASET_ROOT,record[\'relative_directory\'])\n        assert folder.is_dir() and sha256_file(folder/\'manifest.json\')==record[\'manifest_sha256\']\n        manifest=json.loads((folder/\'manifest.json\').read_text())\n        assert manifest[\'status\']==\'COMPLETED\' and manifest[\'schema\']==3 and manifest[\'dataset_id\']==EXPECTED_DATASET_ID\n        assert manifest[\'demo_id\']==demo and manifest[\'dataset_role\']==record[\'dataset_role\']\n        assert manifest[\'source_candidate_fingerprint\']==record[\'source_candidate_fingerprint\']\n        assert manifest[\'export_code_sha256\']==EXPECTED_EXPORT_BUILD and manifest[\'export_protocol_hash\']==EXPECTED_EXPORT_PROTOCOL_HASH\n        assert manifest[\'tail_target_policy\']==CFG[\'tail_target_policy\']\n        assert manifest[\'visual_validation\'][\'all_source_window_reconstruction\']==\'PASS\'\n        assert manifest[\'visual_validation\'][\'historical_prefix_parity\']==\'PASS\'\n        assert manifest[\'visual_validation\'][\'independent_event_max_abs\']<=1e-5\n        role=\'train\' if demo in train_ids else \'validation\'; assert record[\'dataset_role\']==role\n        if role==\'train\': assert NORMALIZATION[\'train_export_manifest_sha256\'][demo]==record[\'manifest_sha256\']\n        pointer=json.loads((DATASET_ROOT/\'paired_exports\'/demo/\'complete.json\').read_text())\n        assert pointer[\'status\']==\'COMPLETED\' and pointer[\'kind\']==\'paired_tail_export\' and pointer[\'schema\']==3 and pointer[\'dataset_id\']==EXPECTED_DATASET_ID and pointer[\'demo_id\']==demo\n        assert safe_child(DATASET_ROOT,pointer[\'relative_directory\'])==folder.resolve() and pointer[\'manifest_sha256\']==record[\'manifest_sha256\']\n        for modality,key in [(\'rgb\',\'rgb_sha256\'),(\'events\',\'events_sha256\')]:\n            name=modality+\'_policy_samples.npz\'\n            assert sha256_file(folder/name)==record[key]==manifest[\'files\'][name]\n        chosen=MODALITY+\'_policy_samples.npz\';other=(\'events\' if MODALITY==\'rgb\' else \'rgb\')+\'_policy_samples.npz\'\n        with np.load(folder/chosen,allow_pickle=False) as z: arrays={k:z[k].copy() for k in z.files}\n        # Companion camera bytes were hashed above; only shared arrays need decompression.\n        with np.load(folder/other,allow_pickle=False) as z:\n            assert set(z.files)==ARRAY_KEYS\n            companion={k:z[k].copy() for k in COMMON_KEYS}; companion.update({k:None for k in IMAGE_KEYS})\n        group=raw[\'data/\'+demo];verify_robot_state_layout(group.attrs[\'model_file\'])\n        n=verify_arrays(arrays,companion,group[\'states\'][:],group[\'actions\'][:]); assert n==record[\'samples\']\n        arrays[\'source_indices\']=np.arange(n,dtype=np.int64);CACHE[demo]=arrays\n        DATA_AUDIT.append({\'demo_id\':demo,\'split\':role,\'samples\':n,\'export_manifest_sha256\':record[\'manifest_sha256\'],\n            \'rgb_sha256\':record[\'rgb_sha256\'],\'events_sha256\':record[\'events_sha256\'],\'source_candidate_fingerprint\':record[\'source_candidate_fingerprint\'],\n            \'endpoint_repeat_slots\':int((~arrays[\'action_is_recorded\']).sum()),\'coverage\':manifest[\'coverage\']})\n        print(demo,\':\',n,\'verified paired\',MODALITY,\'samples\')\n\nprop_mean=np.asarray(NORMALIZATION[\'proprio_mean\'],np.float32);prop_std=np.asarray(NORMALIZATION[\'proprio_std\'],np.float32)\nact_mean=np.asarray(NORMALIZATION[\'action_mean\'],np.float32);act_std=np.asarray(NORMALIZATION[\'action_std\'],np.float32)\nassert prop_mean.shape==prop_std.shape==(36,) and act_mean.shape==act_std.shape==(7,)\nassert all(np.isfinite(a).all() for a in [prop_mean,prop_std,act_mean,act_std])\nassert (prop_std>0).all() and (act_std>0).all()\nfor camera in CAMERAS:\n    mean=np.asarray(NORMALIZATION[\'event_pixel_zscore\'][camera][\'mean\']);std=np.asarray(NORMALIZATION[\'event_pixel_zscore\'][camera][\'std\'])\n    assert mean.shape==std.shape==(3,) and np.isfinite(mean).all() and np.isfinite(std).all() and (std>0).all()\n\ndef config_identity(config):\n    return digest_json({\'code\':CODE_VERSION,\'code_sha256\':CODE_SHA256,\'config\':config,\'data\':DATA_FINGERPRINT,\'normalization\':NORMALIZATION_HASH})\n\ndef run_identity(config):\n    return f"d200_{config[\'modality\']}_paired_gpu_tail_fixed4500_v1_seed{config[\'seed\']}_{config_identity(config)[:12]}"\n\nCONFIG_HASH=config_identity(CFG);RUN_ID=run_identity(CFG)\nRGB_CONFIG=copy.deepcopy(CFG);RGB_CONFIG.update(modality=\'rgb\',visual_normalization=\'fixed_imagenet\')\nREFERENCE_RUN_ID=run_identity(RGB_CONFIG);REFERENCE_CONFIG_HASH=config_identity(RGB_CONFIG)\nRGB_RUN_DIR=PROJECT_ROOT/\'artifacts/checkpoints/rgb\'/REFERENCE_RUN_ID\nRGB_HISTORY=[]\nif MODALITY==\'events\':\n    assert (RGB_RUN_DIR/\'run_config.json\').is_file(), \'Run the NEW Notebook04 paired RGB experiment before Notebook05.\'\n    rgb_record=json.loads((RGB_RUN_DIR/\'run_config.json\').read_text())\n    assert rgb_record[\'run_id\']==REFERENCE_RUN_ID and rgb_record[\'config\']==RGB_CONFIG and rgb_record[\'config_hash\']==REFERENCE_CONFIG_HASH\n    assert rgb_record[\'data_fingerprint\']==DATA_FINGERPRINT and rgb_record[\'normalization_hash\']==NORMALIZATION_HASH\n    rgb_status=json.loads((RGB_RUN_DIR/\'status.json\').read_text());assert rgb_status[\'status\']==\'COMPLETED\' and rgb_status[\'stop_reason\']==\'budget_exhausted\' and rgb_status[\'durable_step\']==CFG[\'max_steps\'], \'Complete the matched fixed-budget RGB run first.\'\n    with open(RGB_RUN_DIR/\'evidence/validation_history.csv\',newline=\'\') as f: rgb_rows=list(csv.DictReader(f))\n    assert rgb_rows and all(r[\'run_id\']==REFERENCE_RUN_ID for r in rgb_rows)\n    RGB_HISTORY=[{\'step\':int(r[\'step\']),\'ema_demo_mean\':float(r[\'ema_demo_mean\']),\'raw_demo_mean\':float(r[\'raw_demo_mean\']), \'ema_common_demo_mean\':float(r[\'ema_common_demo_mean\']), \'raw_common_demo_mean\':float(r[\'raw_common_demo_mean\'])} for r in rgb_rows]\nRUN_DIR=CHECKPOINT_ROOT/RUN_ID;EVIDENCE_DIR=RUN_DIR/\'evidence\';MILESTONE_DIR=RUN_DIR/\'milestones\'\nassert RUN_DIR.resolve().is_relative_to(CHECKPOINT_ROOT.resolve())\nif MODE==\'evaluate\': assert (RUN_DIR/\'run_config.json\').is_file(), \'Evaluation cannot create a new training run.\'\nfor path in [RUN_DIR,EVIDENCE_DIR,MILESTONE_DIR]: path.mkdir(parents=True,exist_ok=True)\nRUN_RECORD={\'run_id\':RUN_ID,\'config_hash\':CONFIG_HASH,\'code_version\':CODE_VERSION,\'code_sha256\':CODE_SHA256,\'config\':CFG,\n    \'data_fingerprint\':DATA_FINGERPRINT,\'paired_sample_fingerprint\':PAIRED_SAMPLE_FINGERPRINT,\'normalization_hash\':NORMALIZATION_HASH,\n    \'dataset_id\':EXPECTED_DATASET_ID,\'modality\':MODALITY,\'rgb_reference_run_id\':REFERENCE_RUN_ID if MODALITY==\'events\' else None}\nif (RUN_DIR/\'run_config.json\').exists():\n    assert json.loads((RUN_DIR/\'run_config.json\').read_text())==RUN_RECORD\n    assert digest_json(json.loads((RUN_DIR/\'normalization.json\').read_text()))==NORMALIZATION_HASH\nelse:\n    atomic_json(RUN_DIR/\'run_config.json\',RUN_RECORD);atomic_json(RUN_DIR/\'normalization.json\',NORMALIZATION)\natomic_json(EVIDENCE_DIR/\'data_audit.json\',{\'dataset_id\':EXPECTED_DATASET_ID,\'data_fingerprint\':DATA_FINGERPRINT,\n    \'raw_hash\':EXPECTED_RAW_SHA256,\'split\':split,\'per_demo\':DATA_AUDIT,\'excluded\':EXCLUSIONS,\n    \'source_protocol\':SOURCE_PROTOCOL,\'normalization_hash\':NORMALIZATION_HASH,\'state_warning\':CFG[\'state_contract\'],\'tail_target_policy\':CFG[\'tail_target_policy\']})\n\nclass PairedDataset(Dataset):\n    def __init__(self,demo_ids,training=False):\n        self.training=training\n        self.samples=[(d,i) for d in demo_ids for i in range(len(CACHE[d][\'control_steps\']))]\n    def get(self,index,crop_mode=None,crop_rng=None):\n        demo,i=self.samples[index];z=CACHE[demo];out={}\n        random_crop=self.training if crop_mode is None else crop_mode==\'random\'\n        rng=random if crop_rng is None else crop_rng;c=CFG[\'crop_size\']\n        for camera in CAMERAS:\n            top=rng.randint(0,84-c) if random_crop else (84-c)//2\n            left=rng.randint(0,84-c) if random_crop else (84-c)//2\n            a=np.stack([z[camera+\'_previous\'][i],z[camera+\'_current\'][i]])\n            if MODALITY==\'rgb\':\n                a=a[:,top:top+c,left:left+c,:].transpose(0,3,1,2).astype(np.float32)/255.\n                mean=NORMALIZATION[\'rgb_image_mean\'];std=NORMALIZATION[\'rgb_image_std\']\n            else:\n                a=a[:,:,top:top+c,left:left+c]\n                mean=NORMALIZATION[\'event_pixel_zscore\'][camera][\'mean\'];std=NORMALIZATION[\'event_pixel_zscore\'][camera][\'std\']\n            a=(a-np.asarray(mean,np.float32)[None,:,None,None])/np.asarray(std,np.float32)[None,:,None,None]\n            out[camera]=torch.from_numpy(np.ascontiguousarray(a,np.float32))\n        p=np.r_[z[\'proprio_previous\'][i],z[\'proprio_current\'][i]]\n        out[\'proprio\']=torch.from_numpy(((p-prop_mean)/prop_std).astype(np.float32))\n        out[\'actions\']=torch.from_numpy(((z[\'action_targets\'][i]-act_mean)/act_std).astype(np.float32))\n        out[\'demo_id\'],out[\'local_index\']=demo,int(z[\'source_indices\'][i])\n        return out\n    def __getitem__(self,index): return self.get(index)\n    def __len__(self): return len(self.samples)\n\nclass BatchStream:\n    """Exact shuffled cursor; no DataLoader prefetch/workers or dropped tail."""\n    def __init__(self, n, seed):\n        self.n, self.rng = n, random.Random(seed)\n        self.order, self.cursor, self.epochs = [], 0, 0\n    def next_indices(self, batch_size):\n        result = []\n        while len(result) < batch_size:\n            if self.cursor == len(self.order):\n                self.order = list(range(self.n))\n                self.rng.shuffle(self.order)\n                self.cursor = 0\n                self.epochs += 1\n            count = min(batch_size-len(result), self.n-self.cursor)\n            result.extend(self.order[self.cursor:self.cursor+count])\n            self.cursor += count\n        return result\n    def state_dict(self):\n        return {\'n\': self.n, \'order\': self.order[:], \'cursor\': self.cursor,\n                \'epochs\': self.epochs, \'rng\': self.rng.getstate()}\n    def load_state_dict(self, state):\n        assert state[\'n\'] == self.n\n        self.order, self.cursor, self.epochs = state[\'order\'][:], state[\'cursor\'], state[\'epochs\']\n        self.rng.setstate(state[\'rng\'])\n\ndef to_device(batch):\n    return {k: v.to(DEVICE) if torch.is_tensor(v) and k != \'local_index\' else v for k, v in batch.items()}\n\ndef batch_from(dataset, indices, crop_mode=None, crop_seed=0):\n    rng = random.Random(crop_seed) if crop_mode is not None else None\n    return to_device(default_collate([dataset.get(i, crop_mode, rng) for i in indices]))\n\n\ntrain_dataset,val_dataset=PairedDataset(train_ids,True),PairedDataset(val_ids,False)\ntrain_eval_dataset=PairedDataset(train_ids,False)\nassert (len(train_dataset),len(val_dataset))==(EXPECTED_TRAIN_SAMPLES,EXPECTED_VAL_SAMPLES)\nprint(\'RUN_ID:\',RUN_ID,\'|\',MODALITY,\'| reviewed NVIDIA source:\',EXPECTED_DATASET_ID)\nprint(\'Samples:\',len(train_dataset),\'train /\',len(val_dataset),\'validation; shared fingerprint:\',DATA_FINGERPRINT)\n\ndef audit_visual_inputs(cache, training_ids, validation_ids):\n    rows=[]\n    for label,ids in [(\'train\',training_ids),(\'validation\',validation_ids)]:\n        for camera in CAMERAS:\n            if MODALITY==\'rgb\': mean=NORMALIZATION[\'rgb_image_mean\'];std=NORMALIZATION[\'rgb_image_std\']\n            else: mean=NORMALIZATION[\'event_pixel_zscore\'][camera][\'mean\'];std=NORMALIZATION[\'event_pixel_zscore\'][camera][\'std\']\n            total=np.zeros(3,np.float64);square=total.copy();active=total.copy();max_abs=total.copy();count=0\n            for demo in ids:\n                for suffix in [\'previous\',\'current\']:\n                    value=cache[demo][camera+\'_\'+suffix]\n                    value=value.transpose(0,3,1,2).astype(np.float64)/255. if MODALITY==\'rgb\' else value.astype(np.float64)\n                    normalized=(value-np.asarray(mean)[:,None,None])/np.asarray(std)[:,None,None]\n                    assert np.isfinite(normalized).all()\n                    count+=value.shape[0]*84*84\n                    total+=normalized.sum((0,2,3));square+=np.square(normalized).sum((0,2,3))\n                    active+=np.count_nonzero(value,axis=(0,2,3));max_abs=np.maximum(max_abs,np.abs(normalized).max((0,2,3)))\n            for channel in range(3):\n                rows.append({\'split\':label,\'camera\':camera,\'channel\':channel,\'modality\':MODALITY,\'pixels\':count,\n                    \'mean_z\':float(total[channel]/count),\'std_z\':float(np.sqrt(max(square[channel]/count-(total[channel]/count)**2,0))),\n                    \'active_fraction\':float(active[channel]/count),\'max_abs_z\':float(max_abs[channel]),\n                    \'scope\':\'uncropped input audit; saved train-only statistics are not refitted\'})\n    return rows\n\natomic_json(EVIDENCE_DIR/\'visual_input_audit.json\',audit_visual_inputs(CACHE,train_ids,val_ids))\n# Preview only a training sample, with unchanged raw modality values.\npreview_demo=train_ids[0];preview=CACHE[preview_demo]\nfig,axes=plt.subplots(2,4 if MODALITY==\'events\' else 2,figsize=(12,6))\nfor row,camera in enumerate(CAMERAS):\n    for col,suffix in enumerate([\'previous\',\'current\']):\n        value=preview[camera+\'_\'+suffix][0]\n        if MODALITY==\'rgb\': axes[row,col].imshow(value);axes[row,col].set_title(camera+\' / \'+suffix)\n        else:\n            for channel in range(3):\n                if suffix==\'current\': axes[row,channel+1].imshow(value[channel],cmap=\'magma\' if channel<2 else \'viridis\');axes[row,channel+1].set_title(camera+\' / \'+CFG[\'event_channels\'][channel])\n            if suffix==\'previous\': axes[row,0].imshow(value[2],cmap=\'viridis\');axes[row,0].set_title(camera+\' / previous time surface\')\nfor ax in axes.flat: ax.axis(\'off\')\nfig.suptitle(\'NEW paired source | \'+MODALITY+\' | \'+preview_demo)\nfig.tight_layout();fig.savefig(EVIDENCE_DIR/\'visual_input_preview.png\',dpi=140);plt.close(fig)\n', '03_model.py': "# Custom project architecture retained; NOT an exact official Diffusion Policy reproduction.\nRESNET_FEATURE_DIM = 512\nPROPRIO_DIM = 2 * CFG['state_per_observation']\nACTION_DIM = CFG['action_dim']\nUNET_DIMS = CFG['unet_dims']\nTIME_EMBED_DIM = CFG['time_embed_dim']\nUNET_KERNEL = CFG['unet_kernel']\nGROUP_NORM_GROUPS = CFG['group_norm_groups']\n\ndef freeze_resnet_except_layer4(backbone):\n    for parameter in backbone.parameters():\n        parameter.requires_grad = False\n    for parameter in backbone.layer4.parameters():\n        parameter.requires_grad = True\n    return backbone\n\n\nclass ResNet18Encoder(nn.Module):\n    def __init__(self):\n        super().__init__()\n        backbone = resnet18(\n            weights=ResNet18_Weights.IMAGENET1K_V1\n        )\n        backbone.fc = nn.Identity()\n        freeze_resnet_except_layer4(backbone)\n        self.backbone = backbone\n        self.output_dim = RESNET_FEATURE_DIM\n        self.train(True)\n\n    def train(self, mode=True):\n        super().train(mode)\n        for module in self.backbone.modules():\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                module.eval()\n        return self\n\n    def forward(self, x):\n        features = self.backbone(x)\n        return features.flatten(1)\n\n\nclass SinusoidalEmbedding(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.dim = dim\n\n    def forward(self, timesteps):\n        half = self.dim // 2\n        device = timesteps.device\n        exponent = (\n            -math.log(10000.0)\n            * torch.arange(half, device=device).float()\n            / max(half - 1, 1)\n        )\n        embedding = (\n            timesteps.float().unsqueeze(1)\n            * torch.exp(exponent).unsqueeze(0)\n        )\n        embedding = torch.cat(\n            [torch.sin(embedding), torch.cos(embedding)],\n            dim=1,\n        )\n        if self.dim % 2:\n            embedding = F.pad(embedding, (0, 1))\n        return embedding\n\n\nclass FiLMResBlock1D(nn.Module):\n    def __init__(self, in_ch, out_ch, cond_dim, kernel_size=5, groups=8):\n        super().__init__()\n        padding = kernel_size // 2\n        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding)\n        self.norm1 = nn.GroupNorm(groups, out_ch)\n        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding)\n        self.norm2 = nn.GroupNorm(groups, out_ch)\n        self.film = nn.Linear(cond_dim, 2 * out_ch)\n        self.skip = (\n            nn.Conv1d(in_ch, out_ch, 1)\n            if in_ch != out_ch\n            else nn.Identity()\n        )\n\n    def forward(self, x, cond):\n        h = self.conv1(x)\n        h = self.norm1(h)\n        h = F.silu(h)\n        h = self.conv2(h)\n        h = self.norm2(h)\n        scale, shift = self.film(cond).chunk(2, dim=1)\n        h = h * (1.0 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)\n        h = F.silu(h)\n        return h + self.skip(x)\n\n\nclass ConditionalUNet1D(nn.Module):\n    def __init__(self, action_dim, cond_dim=256):\n        super().__init__()\n        c1, c2, c3 = UNET_DIMS\n\n        self.time_embed = nn.Sequential(\n            SinusoidalEmbedding(TIME_EMBED_DIM),\n            nn.Linear(TIME_EMBED_DIM, cond_dim),\n            nn.SiLU(),\n            nn.Linear(cond_dim, cond_dim),\n        )\n\n        self.input_proj = nn.Conv1d(action_dim, c1, 1)\n        self.down1 = FiLMResBlock1D(c1, c1, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.downsample1 = nn.Conv1d(c1, c2, 3, stride=2, padding=1)\n        self.down2 = FiLMResBlock1D(c2, c2, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.downsample2 = nn.Conv1d(c2, c3, 3, stride=2, padding=1)\n        self.down3 = FiLMResBlock1D(c3, c3, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.mid = FiLMResBlock1D(c3, c3, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.up2 = nn.Conv1d(c3 + c2, c2, 3, padding=1)\n        self.up2_block = FiLMResBlock1D(c2, c2, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.up1 = nn.Conv1d(c2 + c1, c1, 3, padding=1)\n        self.up1_block = FiLMResBlock1D(c1, c1, cond_dim, UNET_KERNEL, GROUP_NORM_GROUPS)\n        self.out_norm = nn.GroupNorm(GROUP_NORM_GROUPS, c1)\n        self.out = nn.Conv1d(c1, action_dim, 1)\n\n    def forward(self, noisy_actions, condition, timesteps):\n        cond = condition + self.time_embed(timesteps)\n        x = self.input_proj(noisy_actions.transpose(1, 2))\n        skip1 = self.down1(x, cond)\n        x = self.downsample1(skip1)\n        skip2 = self.down2(x, cond)\n        x = self.downsample2(skip2)\n        x = self.down3(x, cond)\n        x = self.mid(x, cond)\n        x = F.interpolate(x, size=skip2.shape[-1], mode='nearest')\n        x = self.up2(torch.cat([x, skip2], dim=1))\n        x = self.up2_block(x, cond)\n        x = F.interpolate(x, size=skip1.shape[-1], mode='nearest')\n        x = self.up1(torch.cat([x, skip1], dim=1))\n        x = self.up1_block(x, cond)\n        x = F.silu(self.out_norm(x))\n        x = self.out(x)\n        return x.transpose(1, 2)\n\n\nclass VisualConditionedDiffusionPolicy(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.agentview_encoder = ResNet18Encoder()\n        self.wrist_encoder = ResNet18Encoder()\n        fusion_in = 4 * RESNET_FEATURE_DIM + PROPRIO_DIM\n        self.condition = nn.Sequential(\n            nn.Linear(fusion_in, 1024),\n            nn.SiLU(),\n            nn.Linear(1024, 256),\n        )\n        self.unet = ConditionalUNet1D(ACTION_DIM, cond_dim=256)\n\n    def encode_observation(self, batch):\n        agent = batch['agentview']\n        wrist = batch['robot0_eye_in_hand']\n        batch_size = agent.shape[0]\n\n        agent_flat = agent.reshape(\n            batch_size * 2, 3, agent.shape[-2], agent.shape[-1]\n        )\n        wrist_flat = wrist.reshape(\n            batch_size * 2, 3, wrist.shape[-2], wrist.shape[-1]\n        )\n\n        agent_features = self.agentview_encoder(agent_flat).reshape(\n            batch_size, 2, -1\n        )\n        wrist_features = self.wrist_encoder(wrist_flat).reshape(\n            batch_size, 2, -1\n        )\n\n        fused = torch.cat(\n            [\n                agent_features[:, 0], agent_features[:, 1],\n                wrist_features[:, 0], wrist_features[:, 1],\n                batch['proprio'],\n            ],\n            dim=1,\n        )\n        return self.condition(fused)\n\n    def predict_noise(self, batch, noisy_actions, timesteps):\n        condition = self.encode_observation(batch)\n        return self.unet(noisy_actions, condition, timesteps)\n\n", '04_diffusion.py': "def cosine_betas(num_steps, max_beta=.999):\n    # Same alpha_bar and cap as diffusers betas_for_alpha_bar / squaredcos_cap_v2.\n    def alpha_bar(t):\n        return math.cos((t + .008) / 1.008 * math.pi / 2) ** 2\n    return np.asarray([min(1-alpha_bar((i+1)/num_steps)/alpha_bar(i/num_steps), max_beta)\n                       for i in range(num_steps)], dtype=np.float64)\n\nclass DDPMScheduler:\n    def __init__(self, device):\n        self.betas = torch.tensor(cosine_betas(CFG['diffusion_steps'], CFG['max_beta']), dtype=torch.float32, device=device)\n        self.alphas = 1-self.betas\n        self.abar = self.alphas.cumprod(0)\n        self.prev = torch.cat([torch.ones(1, device=device), self.abar[:-1]])\n        self.variance = (self.betas * (1-self.prev) / (1-self.abar)).clamp_min(1e-20)\n    def add_noise(self, clean, noise, t):\n        return self.abar[t].sqrt()[:, None, None] * clean + (1-self.abar[t]).sqrt()[:, None, None] * noise\n    def step(self, epsilon, t, x, generator=None):\n        # Full 100-step DDPM, epsilon prediction, fixed_small variance; no x0 clipping.\n        x0 = (x - (1-self.abar[t]).sqrt() * epsilon.float()) / self.abar[t].sqrt()\n        mean = (self.betas[t] * self.prev[t].sqrt() / (1-self.abar[t])) * x0\n        mean = mean + ((1-self.prev[t]) * self.alphas[t].sqrt() / (1-self.abar[t])) * x\n        if t == 0:\n            return x0  # Exact t=0 posterior; avoid FP32 cancellation drift.\n        noise = torch.randn(x.shape, generator=generator, device=x.device, dtype=x.dtype)\n        return mean + self.variance[t].sqrt() * noise\n\nDIFFUSION = DDPMScheduler(DEVICE)\nassert float(DIFFUSION.abar[-1]) < 1e-6, 'Cosine schedule regression: terminal noise must dominate.'\n\ndef lr_factor(completed_steps):\n    # LR used by the NEXT successful update; skip incidents do not advance it.\n    update_number = completed_steps + 1\n    if update_number <= CFG['warmup_steps']:\n        return update_number / CFG['warmup_steps']\n    p = min((update_number-CFG['warmup_steps']) / (CFG['lr_schedule_steps']-CFG['warmup_steps']), 1.)\n    return CFG['min_lr_ratio'] + (1-CFG['min_lr_ratio']) * .5 * (1+math.cos(math.pi*p))\n\ndef make_optimizer(model):\n    encoder, policy = [], []\n    for name, p in model.named_parameters():\n        if p.requires_grad:\n            (encoder if name.startswith(('agentview_encoder', 'wrist_encoder')) else policy).append(p)\n    assert encoder and policy and len({id(p) for p in encoder+policy}) == len(encoder)+len(policy)\n    return torch.optim.AdamW([\n        {'params': policy, 'lr': CFG['policy_lr'], 'name': 'policy'},\n        {'params': encoder, 'lr': CFG['encoder_lr'], 'name': 'encoder'}],\n        betas=tuple(CFG['adam_betas']), eps=CFG['adam_eps'], weight_decay=CFG['weight_decay'], foreach=False)\n\ndef make_scaler():\n    return torch.amp.GradScaler('cuda', enabled=CFG['amp'], init_scale=CFG['initial_amp_scale'])\n\ndef backoff_scaler(scaler):\n    if scaler.is_enabled():\n        scaler.scale(torch.ones((), device=DEVICE))\n        scaler.update(new_scale=max(scaler.get_scale()/2., 1.))\n\ndef autocast(enabled=None):\n    return torch.autocast('cuda', dtype=torch.float16, enabled=CFG['amp'] if enabled is None else enabled)\n\n@torch.no_grad()\ndef update_ema(ema, model, completed_steps):\n    decay = min(CFG['ema_max_decay'], 1-(1+completed_steps/CFG['ema_inv_gamma'])**(-CFG['ema_power']))\n    for dst, src in zip(ema.parameters(), model.parameters()):\n        dst.lerp_(src.detach(), 1-decay)\n    # Buffers are copied, including integer counters. They are never silently left stale.\n    for dst, src in zip(ema.buffers(), model.buffers()):\n        dst.copy_(src)\n    return decay\n\ndef capture_rng():\n    return {'python': random.getstate(), 'numpy': np.random.get_state(),\n            'torch': torch.get_rng_state(), 'cuda': torch.cuda.get_rng_state_all()}\n\ndef restore_rng(state):\n    random.setstate(state['python']); np.random.set_state(state['numpy'])\n    torch.set_rng_state(state['torch']); torch.cuda.set_rng_state_all(state['cuda'])\n\n@contextlib.contextmanager\ndef isolated_rng():\n    state = capture_rng()\n    try:\n        yield\n    finally:\n        restore_rng(state)\n\ndef trainable_finite(model):\n    # Includes EMA parameters (requires_grad=False) and floating point buffers.\n    flags = [torch.isfinite(v).all() for v in model.state_dict().values() if v.is_floating_point()]\n    return bool(torch.stack(flags).all().item())\n\ndef bn_snapshot(model):\n    return {n: b.detach().clone() for n, b in model.named_buffers()}\n\ndef assert_bn_frozen(model, before=None):\n    assert all(not m.training for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm))\n    if before is not None:\n        assert all(torch.equal(before[n], b) for n, b in model.named_buffers()), 'BatchNorm buffers changed.'\n\n@torch.inference_mode()\ndef sample_actions(model, batch, seed=0):\n    model.eval()\n    gen = torch.Generator(device=DEVICE).manual_seed(seed)\n    x = torch.randn((len(batch['actions']), 16, 7), device=DEVICE, generator=gen)\n    with autocast():\n        cond = model.encode_observation(batch)\n    for t in range(CFG['diffusion_steps']-1, -1, -1):\n        ts = torch.full((len(x),), t, device=DEVICE, dtype=torch.long)\n        with autocast():\n            eps = model.unet(x, cond, ts)\n        x = DIFFUSION.step(eps, t, x, gen)\n        assert torch.isfinite(x).all(), f'Non-finite sampled action at diffusion step {t}'\n    return x * torch.tensor(act_std, device=DEVICE) + torch.tensor(act_mean, device=DEVICE)\n\n\ndef model_fingerprint(model):\n    h = hashlib.sha256()\n    for name, value in model.state_dict().items():\n        h.update(name.encode())\n        h.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\n", '05_validation.py': 'def target_stratum_summary(rows):\n    """Descriptive breakdown only; every target still participates in ordinary epsilon MSE."""\n    result={}\n    for label,padded in [(\'fully_recorded\',False),(\'endpoint_repeated\',True)]:\n        selected=[r for r in rows if (r[\'recorded_target_slots\']<16)==padded]\n        demos={}\n        for row in selected:demos.setdefault(row[\'demo_id\'],[]).append(row[\'loss\'])\n        result[label]={\'rows\':len(selected),\'demos\':len(demos),\n            \'demo_mean\':float(np.mean([np.mean(v) for v in demos.values()])) if demos else None}\n    return result\n\ndef stable_seed(*parts):\n    return int(hashlib.sha256(\'|\'.join(map(str, parts)).encode()).hexdigest()[:15], 16)\n\ndef fixed_draw(batch, draw=0, forced_t=None):\n    # One CPU generator per sample. Results do not depend on batch ordering/size.\n    noise, timesteps = [], []\n    for demo, index in zip(batch[\'demo_id\'], batch[\'local_index\'].tolist()):\n        gen = torch.Generator().manual_seed(stable_seed(CFG[\'validation_seed\'], demo, index, draw))\n        noise.append(torch.randn((16, 7), generator=gen))\n        t = int(torch.randint(CFG[\'diffusion_steps\'], (1,), generator=gen)) if forced_t is None else forced_t\n        timesteps.append(t)\n    return torch.stack(noise).to(DEVICE), torch.tensor(timesteps, device=DEVICE, dtype=torch.long)\n\n@torch.inference_mode()\ndef evaluate(model, dataset, indices=None, draws=None, crop_mode=\'center\', use_amp=None, forced_t=None):\n    was_training = model.training\n    model.eval()\n    rows = []\n    indices = list(range(len(dataset))) if indices is None else list(indices)\n    draws = CFG[\'validation_draws\'] if draws is None else draws\n    try:\n        with isolated_rng():\n            for start in range(0, len(indices), CFG[\'eval_batch\']):\n                ids = indices[start:start+CFG[\'eval_batch\']]\n                # Crop RNG is keyed by sample, not batch, for diagnostics too.\n                items = [dataset.get(i, crop_mode, random.Random(stable_seed(CFG[\'diagnostic_seed\'], *dataset.samples[i]))) for i in ids]\n                batch = to_device(default_collate(items))\n                with autocast(use_amp):\n                    condition = model.encode_observation(batch)\n                for draw in range(draws):\n                    noise, ts = fixed_draw(batch, draw, forced_t)\n                    noisy = DIFFUSION.add_noise(batch[\'actions\'], noise, ts)\n                    with autocast(use_amp):\n                        prediction = model.unet(noisy, condition, ts)\n                    losses = (prediction.float()-noise).square().mean((1, 2)).cpu().tolist()\n                    local_ids = batch[\'local_index\'].cpu().tolist()\n                    timestep_ids = ts.cpu().tolist()\n                    if not all(math.isfinite(v) for v in losses):\n                        raise FloatingPointError(\'Non-finite deterministic validation loss\')\n                    for j, value in enumerate(losses):\n                        rows.append({\'demo_id\': batch[\'demo_id\'][j], \'sample\': int(local_ids[j]),\n                                     \'draw\': draw, \'timestep\': int(timestep_ids[j]), \'loss\': value,\n                                     \'recorded_target_slots\':int(CACHE[batch[\'demo_id\'][j]][\'action_is_recorded\'][local_ids[j]].sum())})\n    finally:\n        model.train(was_training)\n    per_demo = {}\n    for row in rows:\n        per_demo.setdefault(row[\'demo_id\'], []).append(row[\'loss\'])\n    per_demo = {d: float(np.mean(v)) for d, v in per_demo.items()}\n    return {\'sample_mean\': float(np.mean([r[\'loss\'] for r in rows])),\n            \'demo_mean\': float(np.mean(list(per_demo.values()))),\n            \'demo_std\': float(np.std(list(per_demo.values()), ddof=1)) if len(per_demo) > 1 else 0.,\n            \'per_demo\': per_demo, \'rows\': rows, \'target_strata\':target_stratum_summary(rows)}\n\ndef evenly_spaced_by_demo(dataset, count):\n    groups = {}\n    for i, (demo, _) in enumerate(dataset.samples):\n        groups.setdefault(demo, []).append(i)\n    return [ids[j] for ids in groups.values() for j in sorted(set(np.linspace(0, len(ids)-1, min(count, len(ids))).astype(int).tolist()))]\n\nTRAIN_DIAG_INDICES = evenly_spaced_by_demo(train_eval_dataset, CFG[\'diagnostic_train_per_demo\'])\nPROFILE_INDICES = evenly_spaced_by_demo(val_dataset, CFG[\'profile_samples_per_val_demo\'])\n\ndef write_csv(path, rows):\n    path = Path(path)\n    keys = list(dict.fromkeys(k for row in rows for k in row))\n    tmp = path.with_name(path.name + \'.tmp\')\n    with open(tmp, \'w\', newline=\'\', encoding=\'utf-8\') as f:\n        if keys:\n            writer = csv.DictWriter(f, fieldnames=keys)\n            writer.writeheader(); writer.writerows(rows)\n    os.replace(tmp, path)\n\nclass EarlyStopping:\n    def __init__(self):\n        self.reference = None\n        self.bad_checks = 0\n    def observe(self, metric, step):\n        if self.reference is None or metric < self.reference-CFG[\'early_stop_min_delta\']:\n            self.reference, self.bad_checks = metric, 0\n        else:\n            self.bad_checks += 1\n        return CFG[\'early_stop_enabled\'] and step >= CFG[\'early_stop_min_step\'] and self.bad_checks >= CFG[\'early_stop_patience\']\n    def state_dict(self):\n        return {\'reference\': self.reference, \'bad_checks\': self.bad_checks}\n    def load_state_dict(self, state):\n        self.reference, self.bad_checks = state[\'reference\'], state[\'bad_checks\']\n\n\ndef common_validation_mean(evaluation):\n    assert all(d in evaluation[\'per_demo\'] for d in COMMON_VAL_IDS)\n    return float(np.mean([evaluation[\'per_demo\'][d] for d in COMMON_VAL_IDS]))\n', '06_checkpoints.py': '# Immutable generations + atomic pointers. Two full resume generations survive a failed save.\nSESSION_ID = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\') + \'_\' + uuid.uuid4().hex[:8]\nSESSION_DIR = EVIDENCE_DIR / \'sessions\' / SESSION_ID\nSESSION_DIR.mkdir(parents=True, exist_ok=True)\natomic_json(SESSION_DIR / \'runtime.json\', RUNTIME)\n\ndef append_event(kind, payload):\n    row = {\'utc\': utc_now(), \'run_id\': RUN_ID, \'session_id\': SESSION_ID, \'kind\': kind, **payload}\n    with open(SESSION_DIR / \'events.jsonl\', \'a\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(row, allow_nan=False, default=json_default) + \'\\n\')\n\ndef atomic_torch_save(value, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    local = Path(\'/content\') / (\'nb05-\' + uuid.uuid4().hex + \'.pt\')\n    remote_tmp = path.with_name(path.name + \'.tmp-\' + uuid.uuid4().hex)\n    try:\n        torch.save(value, local)\n        digest = sha256_file(local)\n        shutil.copyfile(local, remote_tmp)\n        assert sha256_file(remote_tmp) == digest, \'Checkpoint transfer hash mismatch\'\n        os.replace(remote_tmp, path)\n        atomic_json(path.with_suffix(path.suffix + \'.sha256.json\'), {\'sha256\': digest, \'bytes\': path.stat().st_size})\n        return digest\n    finally:\n        local.unlink(missing_ok=True)\n        remote_tmp.unlink(missing_ok=True)\n\ndef load_verified(path):\n    path = Path(path)\n    meta = json.loads(path.with_suffix(path.suffix + \'.sha256.json\').read_text())\n    assert sha256_file(path) == meta[\'sha256\'], f\'Checkpoint checksum mismatch: {path}\'\n    # Trusted files produced by this notebook only. Pickle is not a sandbox.\n    saved = torch.load(path, map_location=\'cpu\', weights_only=False)\n    assert saved[\'run_id\'] == RUN_ID and saved[\'config_hash\'] == CONFIG_HASH\n    assert saved[\'dataset_id\'] == EXPECTED_DATASET_ID and saved[\'paired_normalization_hash\'] == NORMALIZATION_HASH\n    assert saved.get(\'modality\') == MODALITY, \'Checkpoint modality mismatch.\'\n    assert digest_json(saved[\'normalization\']) == digest_json(NORMALIZATION), \'Normalization changed.\'\n    assert saved[\'schema\'] == CFG[\'checkpoint_schema\'] and saved[\'data_fingerprint\'] == DATA_FINGERPRINT\n    return saved\n\ndef checkpoint_candidates(kind):\n    assert kind in [\'best\', \'last\']\n    folder = RUN_DIR / (\'resume\' if kind == \'last\' else \'best\')\n    candidates = list(folder.glob(\'step_*.ckpt\')) if folder.exists() else []\n    # A verified generation may exist even when a disconnect interrupted pointer publication.\n    return sorted(candidates, key=lambda p: (int(p.name.split(\'_\')[1]), p.name), reverse=True)\n\ndef load_latest_verified(kind):\n    candidates = checkpoint_candidates(kind)\n    for path in candidates:\n        try:\n            payload = load_verified(path)\n            return path, payload\n        except (OSError, ValueError, AssertionError, RuntimeError, EOFError) as exc:\n            append_event(\'checkpoint_rejected\', {\'kind_checked\': kind, \'path\': str(path), \'error\': str(exc)})\n            print(\'Rejected checkpoint generation; checking earlier one:\', path.name)\n    if candidates:\n        raise RuntimeError(f\'No verified {kind} checkpoint. Refusing to silently restart this RUN_ID.\')\n    return None, None\n\ndef publish_pointer(kind, path, digest, step):\n    atomic_json(RUN_DIR / f\'{kind}.json\', {\'run_id\': RUN_ID, \'config_hash\': CONFIG_HASH,\n        \'relative_path\': str(path.relative_to(RUN_DIR)), \'sha256\': digest, \'step\': step, \'utc\': utc_now()})\n\ndef cpu_state(model, half=False):\n    return {k: (v.detach().to(\'cpu\', dtype=torch.float16).clone() if half and v.is_floating_point()\n                else v.detach().cpu().clone()) for k, v in model.state_dict().items()}\n\ndef checkpoint_header(step):\n    return {\'modality\': MODALITY, \'dataset_id\': EXPECTED_DATASET_ID, \'paired_normalization_hash\': NORMALIZATION_HASH, \'paired_sample_fingerprint\': PAIRED_SAMPLE_FINGERPRINT, \'schema\': CFG[\'checkpoint_schema\'], \'run_id\': RUN_ID, \'config_hash\': CONFIG_HASH,\n            \'data_fingerprint\': DATA_FINGERPRINT, \'code_version\': CODE_VERSION, \'code_sha256\': CODE_SHA256,\n            \'config\': CFG, \'normalization\': NORMALIZATION, \'global_step\': step, \'utc\': utc_now(), \'runtime\': RUNTIME}\n\ndef save_weights(path, model, ema, step, milestone=False):\n    payload = checkpoint_header(step)\n    payload.update({\'kind\': \'ema_fp16_milestone\' if milestone else \'best_raw_ema_fp32\',\n                    \'ema\': cpu_state(ema, half=milestone)})\n    if not milestone:\n        payload[\'model\'] = cpu_state(model)\n    digest = atomic_torch_save(payload, path)\n    return digest\n\ndef save_best(model, ema, step):\n    path = RUN_DIR / \'best\' / f\'step_{step:06d}_{time.time_ns():020d}_{uuid.uuid4().hex[:8]}.ckpt\'\n    digest = save_weights(path, model, ema, step)\n    publish_pointer(\'best\', path, digest, step)\n    # Keep each strict improvement: an old generation remains an exact FP32 evaluation artifact.\n    return path\n\ndef save_resume(model, ema, optimizer, scaler, stream, early, state):\n    assert trainable_finite(model) and trainable_finite(ema), \'Refusing non-finite checkpoint\'\n    payload = checkpoint_header(state[\'step\'])\n    payload.update({\'kind\': \'full_resume\', \'model\': model.state_dict(), \'ema\': ema.state_dict(),\n                    \'optimizer\': optimizer.state_dict(), \'scaler\': scaler.state_dict(),\n                    \'sampler\': stream.state_dict(), \'rng\': capture_rng(),\n                    \'early_stopping\': early.state_dict(), \'training_state\': state})\n    path = RUN_DIR / \'resume\' / f"step_{state[\'step\']:06d}_{time.time_ns():020d}_{uuid.uuid4().hex[:8]}.ckpt"\n    digest = atomic_torch_save(payload, path)\n    publish_pointer(\'last\', path, digest, state[\'step\'])\n    atomic_json(RUN_DIR / \'status.json\', {\'run_id\': RUN_ID, \'status\': state[\'status\'],\n                \'stop_reason\': state[\'stop_reason\'], \'durable_step\': state[\'step\'],\n                \'best_step\': state[\'best_step\'], \'best_metric\': state[\'best_metric\'], \'utc\': utc_now()})\n    # Prune only this paired D200 run\'s old FULL RESUME generations after successful publication.\n    # All milestones and best weights are retained. No historical directory is modified.\n    others = [p for p in checkpoint_candidates(\'last\') if p != path]\n    # Check the newest predecessor before deleting anything older.\n    keep = []\n    for older in others:\n        try:\n            meta = json.loads(older.with_suffix(older.suffix + \'.sha256.json\').read_text())\n            if sha256_file(older) == meta[\'sha256\']:\n                keep.append(older)\n        except (OSError, ValueError):\n            continue\n        if len(keep) >= CFG[\'resume_generations_to_keep\']-1:\n            break\n    if len(keep) == CFG[\'resume_generations_to_keep\']-1:\n        for older in others:\n            if older not in keep:\n                older.unlink(missing_ok=True)\n                older.with_suffix(older.suffix + \'.sha256.json\').unlink(missing_ok=True)\n\ndef restore_training(model, ema, optimizer, scaler, stream, early, saved):\n    assert saved[\'kind\'] == \'full_resume\', \'Weight-only checkpoint cannot resume training.\'\n    for key in [\'torch\', \'torchvision\', \'cuda\', \'gpu\', \'capability\']:\n        assert saved[\'runtime\'][key] == RUNTIME[key], f\'Runtime changed ({key}); use a new replicate_tag.\'\n    model.load_state_dict(saved[\'model\']); ema.load_state_dict(saved[\'ema\'])\n    optimizer.load_state_dict(saved[\'optimizer\']); scaler.load_state_dict(saved[\'scaler\'])\n    stream.load_state_dict(saved[\'sampler\']); early.load_state_dict(saved[\'early_stopping\'])\n    restore_rng(saved[\'rng\'])\n    return saved[\'training_state\']\n', '07_preflight.py': '# Disposable model: test actual CUDA forward/backward, one AdamW update, EMA,\n# frozen BN buffers, deterministic validation, scheduler, and save/load behavior.\n# Then DELETE everything. Fresh production initialization is in the training cell.\ndef run_preflight():\n    report = {\'utc\': utc_now(), \'run_id\': RUN_ID, \'checks\': {}, \'runtime\': RUNTIME}\n    seed_everything(CFG[\'seed\'])\n    probe = VisualConditionedDiffusionPolicy().to(DEVICE).train()\n    report[\'initial_model_sha256\'] = model_fingerprint(probe)\n    ema_probe = copy.deepcopy(probe).eval().requires_grad_(False)\n    optimizer_probe = make_optimizer(probe)\n    scaler_probe = make_scaler()\n    before_bn = bn_snapshot(probe)\n    assert_bn_frozen(probe)\n    assert all(p.requires_grad == (\'.layer4.\' in n) for n, p in probe.named_parameters()\n               if n.startswith((\'agentview_encoder\', \'wrist_encoder\')))\n    batch = batch_from(train_eval_dataset, range(CFG[\'micro_batch\']), \'center\')\n    assert batch[\'proprio\'].shape == (CFG[\'micro_batch\'], 36)\n    assert batch[\'actions\'].shape == (CFG[\'micro_batch\'], 16, 7)\n    assert batch[\'agentview\'].shape == (CFG[\'micro_batch\'], 2, 3, CFG[\'crop_size\'], CFG[\'crop_size\'])\n    noise, ts = fixed_draw(batch)\n    noisy = DIFFUSION.add_noise(batch[\'actions\'], noise, ts)\n    start = time.perf_counter()\n    optimizer_probe.zero_grad(set_to_none=True)\n    for _ in range(CFG[\'accumulation\']):\n        with autocast():\n            prediction = probe.predict_noise(batch, noisy, ts)\n            loss = F.mse_loss(prediction.float(), noise) / CFG[\'accumulation\']\n        assert torch.isfinite(loss)\n        scaler_probe.scale(loss).backward()\n    scaler_probe.unscale_(optimizer_probe)\n    norm = torch.nn.utils.clip_grad_norm_([p for p in probe.parameters() if p.requires_grad], CFG[\'grad_clip_norm\'], error_if_nonfinite=True)\n    assert float(norm) > 0\n    scaler_probe.step(optimizer_probe); scaler_probe.update()\n    optimizer_probe.zero_grad(set_to_none=True)\n    update_ema(ema_probe, probe, 1)\n    assert_bn_frozen(probe, before_bn)\n    assert all(torch.equal(a, b) for a, b in zip(probe.buffers(), ema_probe.buffers()))\n    assert trainable_finite(probe)\n    torch.cuda.synchronize()\n    report[\'checks\'][\'cuda_backward_optimizer_ema_bn\'] = \'PASS\'\n    report[\'one_optimizer_step_benchmark_seconds\'] = time.perf_counter()-start\n    report[\'benchmark_note\'] = f"One disposable {CFG[\'accumulation\']}x{CFG[\'micro_batch\']} accumulation block; cold start, not sustained throughput."\n    report[\'peak_allocated_gib\'] = torch.cuda.max_memory_allocated()/2**30\n    # Actual model gradients: FP32 batch of 16 vs four chunks of 4.\n    # BN statistics are frozen and GroupNorm acts per sample, so grouping should agree numerically.\n    optimizer_probe.zero_grad(set_to_none=True)\n    with autocast(False):\n        full_loss = F.mse_loss(probe.predict_noise(batch, noisy, ts).float(), noise)\n    full_loss.backward()\n    full_grads = [p.grad.detach().clone() for p in probe.parameters() if p.requires_grad]\n    optimizer_probe.zero_grad(set_to_none=True)\n    for lo in range(0, len(noise), 4):\n        hi = min(lo+4, len(noise))\n        sub = {k: (v[lo:hi] if torch.is_tensor(v) or isinstance(v, list) else v) for k,v in batch.items()}\n        with autocast(False):\n            part_loss = F.mse_loss(probe.predict_noise(sub, noisy[lo:hi], ts[lo:hi]).float(), noise[lo:hi])\n        (part_loss*(hi-lo)/len(noise)).backward()\n    diffs = torch.stack([(p.grad-g).float().square().sum() for p,g in zip(\n        [p for p in probe.parameters() if p.requires_grad], full_grads)]).sum().sqrt()\n    denom = torch.stack([g.float().square().sum() for g in full_grads]).sum().sqrt().clamp_min(1e-12)\n    gradient_relative_error = float(diffs/denom)\n    assert gradient_relative_error < 1e-3, gradient_relative_error\n    report[\'checks\'][\'real_model_microbatch_gradient_agreement\'] = \'PASS\'\n    report[\'gradient_partition_relative_error\'] = gradient_relative_error\n    del full_grads\n    optimizer_probe.zero_grad(set_to_none=True)\n    # Fixed evaluation repeats without advancing global RNG state.\n    rng_before = torch.get_rng_state().clone()\n    a = evaluate(ema_probe, val_dataset, range(4), draws=1)\n    b = evaluate(ema_probe, val_dataset, range(4), draws=1)\n    assert abs(a[\'sample_mean\']-b[\'sample_mean\']) < 1e-7\n    assert torch.equal(rng_before, torch.get_rng_state())\n    report[\'checks\'][\'repeatable_validation\'] = \'PASS\'\n    original_eval_batch = CFG[\'eval_batch\']\n    try:\n        CFG[\'eval_batch\'] = 4\n        small_eval = evaluate(ema_probe, val_dataset, range(min(20,len(val_dataset))), draws=1)\n        CFG[\'eval_batch\'] = original_eval_batch\n        large_eval = evaluate(ema_probe, val_dataset, range(min(20,len(val_dataset))), draws=1)\n        assert abs(small_eval[\'sample_mean\']-large_eval[\'sample_mean\']) < 2e-4\n        report[\'checks\'][\'validation_batch_partition_agreement\'] = \'PASS\'\n    finally:\n        CFG[\'eval_batch\'] = original_eval_batch\n    # Analytic diffusion checks: terminal prior and exact epsilon inversion at t=0.\n    sample0 = DIFFUSION.add_noise(batch[\'actions\'], noise, torch.zeros_like(ts))\n    recovered = DIFFUSION.step(noise, 0, sample0)\n    torch.testing.assert_close(recovered, batch[\'actions\'], rtol=2e-4, atol=2e-4)\n    assert 0 < float(DIFFUSION.abar[-1]) < 1e-6\n    report[\'checks\'][\'ddpm_terminal_and_t0\'] = \'PASS\'\n    # Stateful shuffle roundtrip is tested across an epoch boundary.\n    stream = BatchStream(7, 42)\n    stream.next_indices(6)\n    saved_stream = stream.state_dict()\n    expected = stream.next_indices(11)\n    restored = BatchStream(7, 123); restored.load_state_dict(saved_stream)\n    assert restored.next_indices(11) == expected\n    report[\'checks\'][\'sampler_resume\'] = \'PASS\'\n    # Lightweight real torch serialization/RNG/optimizer test, independent of policy model size.\n    tiny = nn.Linear(2, 1).to(DEVICE)\n    tiny_opt = torch.optim.AdamW(tiny.parameters(), lr=.001)\n    tiny(torch.ones(2, 2, device=DEVICE)).sum().backward(); tiny_opt.step(); tiny_opt.zero_grad()\n    p = Path(\'/content\') / (\'nb05-preflight-\' + uuid.uuid4().hex + \'.pt\')\n    torch.save({\'model\': tiny.state_dict(), \'optimizer\': tiny_opt.state_dict(), \'rng\': capture_rng()}, p)\n    payload = torch.load(p, map_location=\'cpu\', weights_only=False)\n    twin = nn.Linear(2, 1).to(DEVICE); twin.load_state_dict(payload[\'model\'])\n    twin_opt = torch.optim.AdamW(twin.parameters(), lr=.001); twin_opt.load_state_dict(payload[\'optimizer\'])\n    for module, opt in [(tiny, tiny_opt), (twin, twin_opt)]:\n        module(torch.ones(2, 2, device=DEVICE)).sum().backward(); opt.step(); opt.zero_grad()\n    for left, right in zip(tiny.parameters(), twin.parameters()):\n        torch.testing.assert_close(left, right, rtol=0, atol=0)\n    restore_rng(payload[\'rng\']); r1 = torch.randn(4, device=DEVICE)\n    restore_rng(payload[\'rng\']); r2 = torch.randn(4, device=DEVICE)\n    assert torch.equal(r1, r2)\n    p.unlink()\n    report[\'checks\'][\'checkpoint_optimizer_rng_roundtrip\'] = \'PASS\'\n    # Independent arithmetic check for the 1/accumulation loss scaling.\n    tiny_opt.zero_grad(); twin_opt.zero_grad()\n    x = torch.tensor([[1., 2.], [3., -1.], [2., 1.], [-2., 4.]], device=DEVICE)\n    tiny(x).square().mean().backward()\n    for micro_x in x.chunk(2):\n        (twin(micro_x).square().mean()/2).backward()\n    for left, right in zip(tiny.parameters(), twin.parameters()):\n        torch.testing.assert_close(left.grad, right.grad, rtol=2e-5, atol=2e-5)\n    report[\'checks\'][\'gradient_accumulation_scaling\'] = \'PASS\'\n    # Exercise the actual generation publisher/loader with a tiny trusted payload.\n    # Its files are isolated from production RUN_DIR and deleted afterward.\n    global RUN_DIR\n    production_dir = RUN_DIR\n    test_dir = Path(\'/content\') / (\'nb05-checkpoint-test-\' + uuid.uuid4().hex)\n    try:\n        RUN_DIR = test_dir\n        test_dir.mkdir()\n        test_payload = checkpoint_header(1)\n        test_payload.update({\'kind\': \'full_resume\', \'model\': tiny.state_dict(), \'optimizer\': tiny_opt.state_dict()})\n        good = test_dir / \'resume/step_000001_0001.ckpt\'\n        digest = atomic_torch_save(test_payload, good)\n        publish_pointer(\'last\', good, digest, 1)\n        newer = test_dir / \'resume/step_000002_0002.ckpt\'\n        atomic_torch_save(dict(test_payload, global_step=2), newer)\n        newer.write_bytes(b\'interrupted transfer\')\n        recovered_path, recovered = load_latest_verified(\'last\')\n        assert recovered_path == good and recovered[\'global_step\'] == 1\n        report[\'checks\'][\'actual_checkpoint_corruption_fallback\'] = \'PASS\'\n    finally:\n        RUN_DIR = production_dir\n        shutil.rmtree(test_dir, ignore_errors=True)\n    # Force a numerical incident: weights must stay unchanged and scale must recover.\n    tiny_opt.zero_grad()\n    forced_scaler = make_scaler()\n    previous_weights = [p.detach().clone() for p in tiny.parameters()]\n    forced_scaler.scale(tiny(x).square().mean()).backward()\n    next(tiny.parameters()).grad.fill_(float(\'inf\'))\n    forced_scaler.unscale_(tiny_opt)\n    bad_norm = torch.nn.utils.clip_grad_norm_(tiny.parameters(), 1., error_if_nonfinite=False)\n    assert not torch.isfinite(bad_norm)\n    tiny_opt.zero_grad(set_to_none=True)\n    old_scale = forced_scaler.get_scale()\n    backoff_scaler(forced_scaler)\n    assert all(torch.equal(p, q) for p, q in zip(tiny.parameters(), previous_weights))\n    if forced_scaler.is_enabled():\n        assert forced_scaler.get_scale() == max(old_scale/2, 1.)\n    report[\'checks\'][\'nonfinite_skip_and_scale_recovery\'] = \'PASS\'\n    report[\'parameters_total\'] = sum(p.numel() for p in probe.parameters())\n    report[\'parameters_trainable\'] = sum(p.numel() for p in probe.parameters() if p.requires_grad)\n    if CFG[\'unet_dims\'] == [128, 256, 512]:\n        assert report[\'parameters_total\'] == 33894535\n        assert report[\'parameters_trainable\'] == 28328967\n    report[\'parameters_unet\'] = sum(p.numel() for p in probe.unet.parameters())\n    report[\'parameters_encoders\'] = sum(p.numel() for enc in [probe.agentview_encoder, probe.wrist_encoder] for p in enc.parameters())\n    report[\'parameter_count_matches_rgb_run4\'] = True\n    # Two full generations, worst-case best improvement at every validation, all EMA milestones.\n    checkpoints = CFG[\'max_steps\']//CFG[\'validation_every\']\n    full_bytes = report[\'parameters_total\']*8 + report[\'parameters_trainable\']*8\n    report[\'checkpoint_parameter_bytes_upper_gib\'] = (2*full_bytes +\n        report[\'parameters_total\']*(8+2)*checkpoints)/2**30\n    report[\'storage_note\'] = \'Conservative parameter-only budget; add temporary full save and metadata. All best improvements retained.\'\n    return report\n\nresume_preflight_path, resume_preflight_payload = load_latest_verified(\'last\')\nterminal_run = resume_preflight_payload is not None and resume_preflight_payload[\'training_state\'][\'status\'] in [\'COMPLETED\', \'FAILED\']\nif MODE == \'evaluate\' or terminal_run:\n    PREFLIGHT = {\'status\': \'NOT_REQUESTED_NO_TRAINING\', \'parameters_total\': 33894535,\n                 \'parameters_trainable\': 28328967, \'checks\': {}, \'runtime\': RUNTIME}\n    print(\'No disposable optimizer preflight: evaluation or terminal run.\')\nelse:\n    try:\n        PREFLIGHT = run_preflight()\n        PREFLIGHT[\'status\'] = \'PASS\'\n    except Exception as exc:\n        atomic_json(SESSION_DIR / \'preflight_failure.json\', {\'status\': \'FAILED\', \'error\': str(exc), \'traceback\': traceback.format_exc()})\n        raise\n    print(\'Preflight PASS. Disposable models released; production will be rebuilt from seed.\')\ndel resume_preflight_payload\n# No live preflight objects remain in the production training namespace.\ngc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\natomic_json(SESSION_DIR / \'preflight.json\', PREFLIGHT)\nprint(json.dumps(PREFLIGHT, indent=2, default=json_default))\n', '08_evidence.py': "# Historical CPU runs are never used as this new paired comparison reference.\natomic_json(EVIDENCE_DIR/'paired_recipe_audit.json',{\n    'dataset_id':EXPECTED_DATASET_ID,'shared_data_fingerprint':DATA_FINGERPRINT,\n    'shared_normalization_hash':NORMALIZATION_HASH,'modality':MODALITY,\n    'new_rgb_reference_run_id':REFERENCE_RUN_ID if MODALITY=='events' else None,\n    'comparison_scope':'NEW NVIDIA source only; shared architecture/optimizer/noise/selection, modality-specific visual normalization; no task success claim',\n    'historical_checkpoints_loaded':False})\n\ndef export_core(state):\n    write_csv(EVIDENCE_DIR / 'training_progress.csv', state['progress'])\n    write_csv(EVIDENCE_DIR / 'validation_history.csv', state['history'])\n    write_csv(EVIDENCE_DIR / 'numerical_incidents.csv', state['incidents'])\n    write_csv(EVIDENCE_DIR / 'per_demo_history.csv', state['per_demo'])\n    p, h = state['progress'], state['history']\n    fig, axes = plt.subplots(2, 3, figsize=(16, 9))\n    ax = axes[0, 0]\n    if p:\n        ax.plot([r['step'] for r in p], [r['train_loss'] for r in p], label='train / random crop / random noise', alpha=.65)\n    if h:\n        for field, label in [('ema_demo_mean','EMA val / equal demo weight'), ('raw_demo_mean','raw val / equal demo weight'), ('train_diag_ema','EMA train subset / fixed evaluation')]:\n            ax.plot([r['step'] for r in h], [r[field] for r in h], marker='.', label=label)\n    ax.set(title=MODALITY.upper()+' D200: paired denoising objective', ylabel='epsilon MSE'); ax.legend(fontsize=7)\n    ax = axes[0, 1]\n    if p:\n        ax.plot([r['step'] for r in p], [r['grad_norm_mean'] for r in p], label='mean pre-clip norm')\n        ax.plot([r['step'] for r in p], [r['grad_norm_max'] for r in p], alpha=.6, label='max pre-clip norm')\n        ax.plot([r['step'] for r in p], [r['post_clip_norm_mean'] for r in p], ls=':', label='measured mean post-clip norm')\n    ax.axhline(CFG['grad_clip_norm'], color='black', ls='--', label='clip threshold')\n    ax.set(title='Gradient clipping evidence', ylabel='L2 norm'); ax.legend(fontsize=7)\n    ax = axes[0, 2]\n    if p:\n        for field in ['policy_lr','encoder_lr']:\n            ax.plot([r['step'] for r in p], [r[field] for r in p], label=field)\n    ax.set(title='Learning rates used for updates', ylabel='LR'); ax.legend(fontsize=7)\n    ax = axes[1, 0]\n    if p:\n        ax.plot([r['step'] for r in p], [r['clip_fraction'] for r in p], label='fraction of updates clipped')\n    ax.set(title='Clipping frequency', ylabel='fraction', ylim=(0, 1)); ax.legend(fontsize=7)\n    ax = axes[1, 1]\n    if p:\n        ax.plot([r['step'] for r in p], [r['optimizer_steps_per_hour'] for r in p], label='session incl. validation/saves')\n        ax.plot([r['step'] for r in p], [r['training_only_steps_per_hour'] for r in p], label='successful training blocks')\n        ax.legend(fontsize=7)\n    ax.set(title='Session throughput including validation / saves', ylabel='optimizer steps/hour')\n    ax = axes[1, 2]\n    if h:\n        ax.plot([r['step'] for r in h], [r['ema_demo_mean']-r['train_diag_ema'] for r in h], marker='.')\n    ax.axhline(0, color='gray', lw=.7)\n    ax.set(title='EMA generalization gap (fixed protocol)', ylabel='val - train subset MSE')\n    for ax in axes.flat:\n        ax.set_xlabel('successful optimizer steps'); ax.grid(alpha=.2)\n    fig.suptitle(RUN_ID, fontsize=11); fig.tight_layout()\n    fig.savefig(EVIDENCE_DIR / 'training_dashboard.png', dpi=180)\n    fig.savefig(EVIDENCE_DIR / 'training_dashboard.pdf'); plt.close(fig)\n    fig,axes = plt.subplots(1,2,figsize=(13,4.5))\n    if RGB_HISTORY: axes[0].plot([r['step'] for r in RGB_HISTORY],[r['ema_demo_mean'] for r in RGB_HISTORY],marker='.',label='NEW paired RGB EMA')\n    if h:axes[0].plot([r['step'] for r in h],[r['ema_demo_mean'] for r in h],marker='.',label=MODALITY.upper()+' EMA')\n    axes[0].set(title='Full validation: same fixed noise and demo weights',ylabel='epsilon MSE')\n    if p:axes[1].plot([r['step'] for r in p],[r['train_loss'] for r in p],label=MODALITY.upper()+' training / random noise')\n    if h:axes[1].plot([r['step'] for r in h],[r['train_diag_ema'] for r in h],label=MODALITY.upper()+' train subset / fixed noise')\n    axes[1].set(title='Different noise/crop protocols are labelled',ylabel='epsilon MSE')\n    for ax in axes:ax.set_xlabel('successful optimizer steps');ax.grid(alpha=.2);ax.legend(fontsize=8)\n    fig.suptitle('RGB vs Events D200 development denoising; not task success',fontsize=11)\n    fig.tight_layout();fig.savefig(EVIDENCE_DIR/'rgb_events_comparison.png',dpi=170)\n    fig.savefig(EVIDENCE_DIR/'rgb_events_comparison.pdf');plt.close(fig)\n    # The common ten are a reporting subset of the full evaluation, never a refit.\n    fig,axes=plt.subplots(1,2,figsize=(13,4.5))\n    for rows,label in [(RGB_HISTORY,'matched RGB reference'),(h,MODALITY.upper())]:\n        if not rows:continue\n        xs=[r['step'] for r in rows]\n        axes[0].plot(xs,[r['ema_demo_mean'] for r in rows],marker='.',label=label+' full40 EMA')\n        axes[1].plot(xs,[r['ema_common_demo_mean'] for r in rows],marker='.',label=label+' common10 EMA')\n    for ax,title in zip(axes,['Full D200 validation: 40 demos','Original held-out subset: 10 demos']):\n        ax.set(title=title,xlabel='successful optimizer updates',ylabel='epsilon MSE');ax.grid(alpha=.2);ax.legend(fontsize=8)\n    fig.suptitle('Same D200 normalization; denoising curves are not rollout success')\n    fig.tight_layout();fig.savefig(EVIDENCE_DIR/'validation_full_common.png',dpi=170)\n    fig.savefig(EVIDENCE_DIR/'validation_full_common.pdf');plt.close(fig)\n\ndef export_efficiency(state):\n    timing = []\n    for log in sorted((EVIDENCE_DIR / 'sessions').glob('*/events.jsonl')):\n        for line in log.read_text().splitlines():\n            try:\n                event = json.loads(line)\n                if event.get('kind') == 'cycle_timing':\n                    timing.append(event)\n            except ValueError:\n                continue  # a killed runtime can leave only its final JSONL line incomplete\n    write_csv(EVIDENCE_DIR / 'cycle_timing.csv', timing)\n    progress = state['progress']\n    if not progress:\n        return\n    fig, axes = plt.subplots(2, 2, figsize=(13, 8))\n    steps = [r['step'] for r in progress]\n    axes[0,0].plot(steps, [r['gpu_peak_allocated_gib'] for r in progress], label='peak allocated')\n    axes[0,0].plot(steps, [r['gpu_reserved_gib'] for r in progress], label='reserved')\n    axes[0,0].set(title='CUDA memory', ylabel='GiB'); axes[0,0].legend()\n    axes[0,1].plot(steps, [r['amp_scale'] for r in progress], label='GradScaler scale')\n    axes[0,1].set(title='AMP numerical scale', ylabel='scale'); axes[0,1].set_yscale('log')\n    axes[1,0].plot(steps, [r['grad_norm_over_clip'] for r in progress])\n    axes[1,0].axhline(1,ls='--',color='gray')\n    axes[1,0].set(title='Pre-clip norm / clipping threshold', ylabel='ratio (not gradient normalization)')\n    if timing:\n        last_session = timing[-1]['session_id']\n        rows = [r for r in timing if r['session_id'] == last_session]\n        xs = [r['step'] for r in rows]\n        for key in ['validation_seconds','checkpoint_seconds','evidence_seconds']:\n            axes[1,1].plot(xs, [r[key] for r in rows], marker='.', label=key)\n        axes[1,1].legend(fontsize=7)\n    axes[1,1].set(title='Last session: validation/save/plot costs', ylabel='seconds per validation cycle')\n    for ax in axes.flat:\n        ax.set_xlabel('successful optimizer steps'); ax.grid(alpha=.2)\n    fig.suptitle(RUN_ID, fontsize=10); fig.tight_layout()\n    fig.savefig(EVIDENCE_DIR / 'efficiency_dashboard.png', dpi=180)\n    fig.savefig(EVIDENCE_DIR / 'efficiency_dashboard.pdf'); plt.close(fig)\n", '09_train.py': 'def train_run():\n    resume_path, resume_payload = load_latest_verified(\'last\')\n    if MODE == \'evaluate\':\n        assert resume_payload is not None, \'Evaluation requires an existing same-ID paired checkpoint.\'\n        print(\'Evaluation mode: no optimizer created, no training updates.\')\n        return resume_payload[\'training_state\']\n    if resume_payload is not None and resume_payload[\'training_state\'][\'status\'] in [\'COMPLETED\', \'FAILED\']:\n        print(\'Terminal PAIRED D200 TAIL; optimizer loop will not restart.\')\n        return resume_payload[\'training_state\']\n    assert MODE == \'train\'\n    # Critical reset AFTER all disposable probes: fresh ImageNet encoders / fresh random U-Net.\n    seed_everything(CFG[\'seed\'])\n    model = VisualConditionedDiffusionPolicy().to(DEVICE)\n    assert model_fingerprint(model) == PREFLIGHT[\'initial_model_sha256\'], \'Preflight contaminated fresh initialization.\'\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer, scaler = make_optimizer(model), make_scaler()\n    stream = BatchStream(len(train_dataset), CFG[\'seed\'])\n    early = EarlyStopping()\n    state = {\'step\': 0, \'best_step\': None, \'best_metric\': None, \'status\': \'RUNNING\',\n             \'stop_reason\': None, \'started_utc\': utc_now(), \'ended_utc\': None,\n             \'progress\': [], \'history\': [], \'per_demo\': [], \'incidents\': [],\n             \'consecutive_nonfinite\': 0, \'attempts\': 0, \'successful_microbatches\': 0,\n             \'samples_consumed\': 0, \'training_runtime\': copy.deepcopy(RUNTIME)}\n    if resume_payload is not None:\n        state = restore_training(model, ema, optimizer, scaler, stream, early, resume_payload)\n        del resume_payload\n        print(\'Resumed SAME PAIRED D200 TAIL ID from durable step\', state[\'step\'])\n    else:\n        assert all(torch.equal(a, b) for a, b in zip(model.parameters(), ema.parameters()))\n        print(\'Fresh PAIRED D200 TAIL from ImageNet; policy random initialization. No old checkpoint loaded.\')\n    if state[\'status\'] in [\'COMPLETED\', \'FAILED\']:\n        print(\'Terminal run:\', state[\'status\'], state[\'stop_reason\'], \'- training will not restart.\')\n        return state\n    state[\'status\'], state[\'stop_reason\'] = \'RUNNING\', None\n    model.train(); assert_bn_frozen(model)\n    fixed_bn = bn_snapshot(model)\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    print(\'=== PAIRED D200 TAIL TRAINING ===\', MODALITY, RUN_ID)\n    print(\'LR:\', CFG[\'policy_lr\'], \'/\', CFG[\'encoder_lr\'], \'| effective batch:\', CFG[\'micro_batch\']*CFG[\'accumulation\'])\n    print(\'Max steps:\', CFG[\'max_steps\'], \'| LR schedule horizon:\', CFG[\'lr_schedule_steps\'], \'| validation:\', CFG[\'validation_every\'], \'| robot-only state:\', PROPRIO_DIM)\n    print(\'Denoising early stopping enabled:\', CFG[\'early_stop_enabled\'], \'| common validation demos:\', len(COMMON_VAL_IDS))\n    append_event(\'training_start\', {\'step\': state[\'step\'], \'config_hash\': CONFIG_HASH})\n    if resume_path is None:\n        # A step-0 recovery point exists even if the first 500 updates are interrupted.\n        save_resume(model, ema, optimizer, scaler, stream, early, state)\n    session_start, session_first_step = time.perf_counter(), state[\'step\']\n    window_losses, window_norms, window_post_norms = [], [], []\n    training_seconds = 0.0\n    window_training_seconds = 0.0\n    unsafe_exception = False\n    try:\n        while state[\'step\'] < CFG[\'max_steps\']:\n            state[\'attempts\'] += 1\n            update_start = time.perf_counter()\n            optimizer.zero_grad(set_to_none=True)\n            factor = lr_factor(state[\'step\'])\n            for group, lr in zip(optimizer.param_groups, [CFG[\'policy_lr\'], CFG[\'encoder_lr\']]):\n                group[\'lr\'] = lr * factor\n            block_loss, issue = torch.zeros((), device=DEVICE), None\n            all_losses_finite = torch.ones((), device=DEVICE, dtype=torch.bool)\n            for micro in range(CFG[\'accumulation\']):\n                batch = batch_from(train_dataset, stream.next_indices(CFG[\'micro_batch\']))\n                state[\'samples_consumed\'] += len(batch[\'actions\'])\n                clean = batch[\'actions\']\n                noise = torch.randn_like(clean)\n                ts = torch.randint(CFG[\'diffusion_steps\'], (len(clean),), device=DEVICE)\n                noisy = DIFFUSION.add_noise(clean, noise, ts)\n                with autocast():\n                    prediction = model.predict_noise(batch, noisy, ts)\n                    loss = F.mse_loss(prediction.float(), noise)\n                all_losses_finite.logical_and_(torch.isfinite(loss.detach()))\n                scaler.scale(loss / CFG[\'accumulation\']).backward()\n                block_loss += loss.detach() / CFG[\'accumulation\']\n            # One host read per accumulation block. A nonfinite backward only affects disposable gradients.\n            scaler.unscale_(optimizer)\n            norm = torch.nn.utils.clip_grad_norm_(trainable, CFG[\'grad_clip_norm\'], error_if_nonfinite=False)\n            post_norm = torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(p.grad.detach().float())\n                         for p in trainable if p.grad is not None]))\n            block_loss_value, norm_value, post_norm_value, losses_ok = torch.stack(\n                [block_loss, norm.float(), post_norm.float(), all_losses_finite.float()]).detach().cpu().tolist()\n            if not losses_ok:\n                issue = \'nonfinite_loss\'\n            elif not math.isfinite(norm_value) or not math.isfinite(post_norm_value):\n                issue = \'nonfinite_gradient_norm\'\n            if issue is not None:\n                # Entire accumulation block discarded. No optimizer/EMA/LR/global-step advancement.\n                optimizer.zero_grad(set_to_none=True)\n                before_scale = float(scaler.get_scale())\n                backoff_scaler(scaler)\n                state[\'consecutive_nonfinite\'] += 1\n                row = {\'run_id\': RUN_ID, \'step\': state[\'step\'], \'attempt\': state[\'attempts\'],\n                       \'kind\': issue, \'scale_before\': before_scale, \'scale_after\': float(scaler.get_scale()),\n                       \'consecutive\': state[\'consecutive_nonfinite\'], \'utc\': utc_now()}\n                state[\'incidents\'].append(row); append_event(\'nonfinite\', row)\n                print(\'Skipped unsafe accumulation block:\', row)\n                if (state[\'consecutive_nonfinite\'] >= CFG[\'max_consecutive_nonfinite\'] or\n                        len(state[\'incidents\']) >= CFG[\'max_total_nonfinite\']):\n                    state[\'status\'], state[\'stop_reason\'] = \'FAILED\', \'nonfinite_limit\'\n                    state[\'ended_utc\'] = utc_now()\n                    save_resume(model, ema, optimizer, scaler, stream, early, state)\n                    break\n                continue\n            scaler.step(optimizer); scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n            state[\'step\'] += 1\n            state[\'successful_microbatches\'] += CFG[\'accumulation\']\n            state[\'consecutive_nonfinite\'] = 0\n            decay = update_ema(ema, model, state[\'step\'])\n            window_losses.append(block_loss_value); window_norms.append(norm_value)\n            window_post_norms.append(post_norm_value)\n            # Synchronize once per optimizer update so EMA time is charged to training.\n            torch.cuda.synchronize()\n            update_seconds = time.perf_counter()-update_start\n            training_seconds += update_seconds; window_training_seconds += update_seconds\n            step = state[\'step\']\n            if step % CFG[\'evidence_every\'] == 0:\n                elapsed = time.perf_counter()-session_start\n                row = {\'run_id\': RUN_ID, \'step\': step, \'train_loss\': float(np.mean(window_losses)),\n                       \'grad_norm_mean\': float(np.mean(window_norms)), \'grad_norm_max\': max(window_norms),\n                       \'post_clip_norm_mean\': float(np.mean(window_post_norms)),\n                       \'grad_norm_over_clip\': float(np.mean(window_norms))/CFG[\'grad_clip_norm\'],\n                       \'clip_fraction\': float(np.mean(np.array(window_norms) > CFG[\'grad_clip_norm\'])),\n                       \'policy_lr\': optimizer.param_groups[0][\'lr\'], \'encoder_lr\': optimizer.param_groups[1][\'lr\'],\n                       \'ema_decay\': decay, \'amp_scale\': float(scaler.get_scale()),\n                       \'optimizer_steps_per_hour\': (step-session_first_step)/elapsed*3600,\n                       \'training_only_steps_per_hour\': (step-session_first_step)/max(training_seconds, 1e-9)*3600,\n                       \'window_training_steps_per_hour\': len(window_losses)/max(window_training_seconds, 1e-9)*3600,\n                       \'window_training_sec_per_update\': window_training_seconds/max(len(window_losses),1),\n                       \'session_elapsed_h\': elapsed/3600, \'data_epochs_started\': stream.epochs,\n                       \'successful_sample_exposures\': step*64, \'samples_consumed\': state[\'samples_consumed\'],\n                       \'equivalent_data_passes\': step*64/len(train_dataset),\n                       \'gpu_peak_allocated_gib\': torch.cuda.max_memory_allocated()/2**30,\n                       \'gpu_reserved_gib\': torch.cuda.memory_reserved()/2**30,\n                       \'nonfinite_total\': len(state[\'incidents\'])}\n                state[\'progress\'].append(row); append_event(\'progress\', row)\n                print(f"step={step} | train={row[\'train_loss\']:.6f} | grad={row[\'grad_norm_mean\']:.4f} | "\n                      f"lr={row[\'policy_lr\']:.3e} | clipped={row[\'clip_fraction\']:.0%} | "\n                      f"{row[\'optimizer_steps_per_hour\']:.0f} steps/h")\n                window_losses, window_norms, window_post_norms = [], [], []\n                window_training_seconds = 0.0\n            if step % CFG[\'validation_every\'] == 0 or step == CFG[\'max_steps\']:\n                assert_bn_frozen(model, fixed_bn)\n                validation_start = time.perf_counter()\n                ema_eval = evaluate(ema, val_dataset)\n                raw_eval = evaluate(model, val_dataset)\n                train_eval = evaluate(ema, train_eval_dataset, TRAIN_DIAG_INDICES)\n                validation_seconds = time.perf_counter()-validation_start\n                checkpoint_start = time.perf_counter()\n                metric = ema_eval[\'demo_mean\']\n                improved = state[\'best_metric\'] is None or metric < state[\'best_metric\']\n                if improved:\n                    state[\'best_metric\'], state[\'best_step\'] = metric, step\n                    save_best(model, ema, step)\n                stop_early = early.observe(metric, step)\n                row = {\'run_id\': RUN_ID, \'step\': step, \'ema_demo_mean\': metric,\n                       \'raw_demo_mean\': raw_eval[\'demo_mean\'],\n                       \'ema_common_demo_mean\': common_validation_mean(ema_eval),\n                       \'raw_common_demo_mean\': common_validation_mean(raw_eval),\n                       \'ema_sample_mean\': ema_eval[\'sample_mean\'],\n                       \'raw_sample_mean\': raw_eval[\'sample_mean\'], \'ema_demo_std\': ema_eval[\'demo_std\'],\n                       \'train_diag_ema\': train_eval[\'demo_mean\'], \'best_metric\': state[\'best_metric\'],\n                       \'best_step\': state[\'best_step\'], \'bad_checks\': early.bad_checks,\n                       \'validation_seconds\': validation_seconds,\n                       \'ema_fully_recorded_demo_mean\':ema_eval.get(\'target_strata\',{}).get(\'fully_recorded\',{}).get(\'demo_mean\'),\n                       \'ema_endpoint_repeated_demo_mean\':ema_eval.get(\'target_strata\',{}).get(\'endpoint_repeated\',{}).get(\'demo_mean\')}\n                state[\'history\'].append(row); append_event(\'validation\', row)\n                for d in val_ids:\n                    state[\'per_demo\'].append({\'run_id\': RUN_ID, \'step\': step, \'demo_id\': d,\n                                             \'ema_loss\': ema_eval[\'per_demo\'][d], \'raw_loss\': raw_eval[\'per_demo\'][d]})\n                write_csv(EVIDENCE_DIR / f\'validation_samples_step_{step:06d}.csv\', ema_eval[\'rows\'])\n                if stop_early or step == CFG[\'max_steps\']:\n                    state[\'status\'] = \'COMPLETED\'\n                    state[\'stop_reason\'] = \'early_stopping\' if stop_early else \'budget_exhausted\'\n                    state[\'ended_utc\'] = utc_now()\n                save_weights(MILESTONE_DIR / f\'step_{step:06d}_ema_fp16.ckpt\', model, ema, step, milestone=True)\n                save_resume(model, ema, optimizer, scaler, stream, early, state)\n                checkpoint_seconds = time.perf_counter()-checkpoint_start\n                export_start = time.perf_counter()\n                export_core(state)\n                append_event(\'cycle_timing\', {\'step\': step, \'validation_seconds\': validation_seconds,\n                    \'checkpoint_seconds\': checkpoint_seconds, \'evidence_seconds\': time.perf_counter()-export_start,\n                    \'session_wall_seconds\': time.perf_counter()-session_start})\n                print(f"VALIDATION step={step} | EMA demo={metric:.6f} | raw={raw_eval[\'demo_mean\']:.6f} | "\n                      f"common10={row[\'ema_common_demo_mean\']:.6f} | best={state[\'best_metric\']:.6f}@{state[\'best_step\']} | early_stop_enabled={CFG[\'early_stop_enabled\']}")\n                print(\'Saved best.json/last.json + verified generations/milestone + CSV/graphs:\', RUN_DIR)\n                if state[\'status\'] == \'COMPLETED\':\n                    break\n    except KeyboardInterrupt:\n        # Interrupt may land inside optimizer.step. Never serialize uncertain live weights.\n        unsafe_exception = True\n        state[\'status\'], state[\'stop_reason\'], state[\'ended_utc\'] = \'ABORTED\', \'keyboard_interrupt\', utc_now()\n        append_event(\'interrupted\', {\'logged_step\': state[\'step\'], \'reason\': \'use_previous_durable_last_only\'})\n        print(\'Interrupted. Last completed checkpoint is preserved; up to 499 updates may need replay.\')\n    except Exception as exc:\n        unsafe_exception = True\n        state[\'status\'], state[\'stop_reason\'], state[\'ended_utc\'] = \'FAILED\', type(exc).__name__, utc_now()\n        append_event(\'exception\', {\'logged_step\': state[\'step\'], \'error\': str(exc), \'traceback\': traceback.format_exc()})\n        print(\'TRAINING FAILED:\', repr(exc), \'\\nPrevious durable checkpoints remain available.\')\n    finally:\n        optimizer.zero_grad(set_to_none=True)\n        durable = json.loads((RUN_DIR / \'status.json\').read_text()).get(\'durable_step\', 0) if (RUN_DIR / \'status.json\').exists() else 0\n        atomic_json(SESSION_DIR / \'result.json\', {\'run_id\': RUN_ID, \'status\': state[\'status\'],\n                    \'stop_reason\': state[\'stop_reason\'], \'observed_step\': state[\'step\'], \'durable_step\': durable,\n                    \'unsafe_live_weights_discarded\': unsafe_exception, \'utc\': utc_now(),\n                    \'training_seconds_this_session\': training_seconds,\n                    \'wall_seconds_this_session\': time.perf_counter()-session_start})\n        export_core(state)\n    return state\n\nTRAINING_STATE = train_run()\ngc.collect(); torch.cuda.empty_cache()\nprint(\'Training cell ended:\', TRAINING_STATE[\'status\'], TRAINING_STATE[\'stop_reason\'])\n', '10_diagnostics.py': 'def condition_donor_map(dataset):\n    groups = {}\n    for i, (demo, _) in enumerate(dataset.samples):\n        groups.setdefault(demo, []).append(i)\n    demos = sorted(groups)\n    assert len(demos) > 1\n    donors = {}\n    for d, demo in enumerate(demos):\n        recipient, donor = groups[demo], groups[demos[(d+1)%len(demos)]]\n        for j, index in enumerate(recipient):\n            donors[index] = donor[min(int(j*len(donor)/len(recipient)),len(donor)-1)]\n    assert all(dataset.samples[i][0] != dataset.samples[j][0] for i,j in donors.items())\n    return donors\n\n@torch.inference_mode()\ndef condition_sensitivity(model, label, step):\n    """Fixed clean actions/noise, mismatched observations from a different validation demo.\n    These counterfactual inputs may be off-distribution; sensitivity is not rollout competence.\n    """\n    model.eval()\n    donors = condition_donor_map(val_dataset)\n    rows = []\n    for start in range(0,len(PROFILE_INDICES),CFG[\'eval_batch\']):\n        ids = PROFILE_INDICES[start:start+CFG[\'eval_batch\']]\n        base = batch_from(val_dataset, ids, \'center\')\n        donor = batch_from(val_dataset, [donors[i] for i in ids], \'center\')\n        noise, _ = fixed_draw(base, draw=0)\n        for mode in [\'matched\', \'vision_swapped\', \'proprio_swapped\', \'both_swapped\']:\n            altered = dict(base)\n            if mode in [\'vision_swapped\',\'both_swapped\']:\n                for camera in [\'agentview\',\'robot0_eye_in_hand\']:\n                    altered[camera] = donor[camera]\n            if mode in [\'proprio_swapped\',\'both_swapped\']:\n                altered[\'proprio\'] = donor[\'proprio\']\n            with autocast():\n                condition = model.encode_observation(altered)\n            for t in CFG[\'condition_probe_timesteps\']:\n                ts = torch.full((len(noise),), t, device=DEVICE, dtype=torch.long)\n                noisy = DIFFUSION.add_noise(base[\'actions\'], noise, ts)\n                with autocast():\n                    pred = model.unet(noisy, condition, ts)\n                errors = (pred.float()-noise).square().mean((1,2)).cpu().tolist()\n                assert all(math.isfinite(e) for e in errors)\n                for j,err in enumerate(errors):\n                    rows.append({\'run_id\':RUN_ID, \'checkpoint\':label, \'step\':step,\n                        \'mode\':mode, \'timestep\':t, \'demo_id\':base[\'demo_id\'][j],\n                        \'sample\':int(base[\'local_index\'][j]), \'donor_demo\':donor[\'demo_id\'][j], \'loss\':err})\n    return rows\n\n@torch.inference_mode()\ndef timestep_profile(model, label, checkpoint_step):\n    model.eval()\n    rows = []\n    for start in range(0, len(PROFILE_INDICES), CFG[\'eval_batch\']):\n        batch = batch_from(val_dataset, PROFILE_INDICES[start:start+CFG[\'eval_batch\']], \'center\')\n        with autocast():\n            condition = model.encode_observation(batch)\n        noise, _ = fixed_draw(batch, draw=0)\n        for timestep in range(CFG[\'diffusion_steps\']):\n            ts = torch.full((len(noise),), timestep, device=DEVICE, dtype=torch.long)\n            noisy = DIFFUSION.add_noise(batch[\'actions\'], noise, ts)\n            with autocast():\n                prediction = model.unet(noisy, condition, ts)\n            losses = (prediction.float()-noise).square().mean((1, 2)).cpu().tolist()\n            assert all(math.isfinite(v) for v in losses)\n            for j, loss in enumerate(losses):\n                rows.append({\'run_id\': RUN_ID, \'checkpoint\': label, \'step\': checkpoint_step,\n                             \'demo_id\': batch[\'demo_id\'][j], \'sample\': int(batch[\'local_index\'][j]),\n                             \'timestep\': timestep, \'loss\': loss})\n    return rows\n\ndef final_diagnostics():\n    summary = {\'run_id\': RUN_ID, \'status\': \'RUNNING\', \'utc\': utc_now(), \'checkpoints\': {}}\n    profiles, demo_rows, sensitivity_rows = [], [], []\n    labels = [\'best\', \'last\']\n    seen_steps = set()\n    for label in labels:\n        path, saved = load_latest_verified(label)\n        if saved is None:\n            summary[\'checkpoints\'][label] = {\'status\': \'MISSING\'}\n            continue\n        step = saved[\'global_step\']\n        if step in seen_steps:\n            summary[\'checkpoints\'][label] = {\'status\': \'SAME_STEP_AS_BEST\', \'step\': step}\n            del saved\n            continue\n        seen_steps.add(step)\n        model = VisualConditionedDiffusionPolicy().to(DEVICE).eval()\n        model.load_state_dict(saved[\'model\'])\n        raw_metrics = evaluate(model, val_dataset)\n        model.load_state_dict(saved[\'ema\'])\n        del saved\n        ema_metrics = evaluate(model, val_dataset)\n        train_metrics = evaluate(model, train_eval_dataset, TRAIN_DIAG_INDICES)\n        center = evaluate(model, val_dataset, PROFILE_INDICES, draws=1)\n        random_crop = evaluate(model, val_dataset, PROFILE_INDICES, draws=1, crop_mode=\'random\')\n        fp32 = evaluate(model, val_dataset, PROFILE_INDICES, draws=1, use_amp=False)\n        per_demo = np.asarray(list(ema_metrics[\'per_demo\'].values()))\n        rng = np.random.default_rng(CFG[\'diagnostic_seed\'])\n        means = per_demo[rng.integers(0, len(per_demo), (2000, len(per_demo)))].mean(1)\n        details = {\'status\': \'COMPLETED\', \'step\': step, \'raw_demo_mean\': raw_metrics[\'demo_mean\'],\n                   \'ema_demo_mean\': ema_metrics[\'demo_mean\'], \'ema_sample_mean\': ema_metrics[\'sample_mean\'],\n                   \'ema_target_strata\':ema_metrics[\'target_strata\'],\'raw_target_strata\':raw_metrics[\'target_strata\'],\n                   \'ema_common_demo_mean\': common_validation_mean(ema_metrics),\n                   \'raw_common_demo_mean\': common_validation_mean(raw_metrics),\n                   \'ema_train_subset_demo_mean\': train_metrics[\'demo_mean\'],\n                   \'descriptive_demo_bootstrap_95pct\': np.quantile(means, [.025, .975]).tolist(),\n                   \'ci_scope\': f\'{len(val_ids)} held-out demos, fixed trained model/noise; not training-seed or rollout uncertainty\',\n                   \'crop_subset_center\': center[\'demo_mean\'], \'crop_subset_random\': random_crop[\'demo_mean\'],\n                   \'amp_subset\': center[\'demo_mean\'], \'fp32_subset\': fp32[\'demo_mean\'],\n                   \'profile_samples\': len(PROFILE_INDICES), \'profile_draws\': 1, \'profile_timesteps\': 100,\n                   \'checkpoint_path\': str(path), \'checkpoint_sha256\': sha256_file(path),\n                   \'config_hash\': CONFIG_HASH, \'data_fingerprint\': DATA_FINGERPRINT}\n        for d in val_ids:\n            demo_rows.append({\'run_id\': RUN_ID, \'checkpoint\': label, \'step\': step, \'demo_id\': d,\n                             \'ema_loss\': ema_metrics[\'per_demo\'][d], \'raw_loss\': raw_metrics[\'per_demo\'][d]})\n        profiles.extend(timestep_profile(model, label, step))\n        sensitivity_rows.extend(condition_sensitivity(model, label, step))\n        summary[\'checkpoints\'][label] = details\n        atomic_json(EVIDENCE_DIR / \'final_diagnostics.json\', summary)\n        print(label, \'@\', step, details)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n    write_csv(EVIDENCE_DIR / \'timestep_profile.csv\', profiles)\n    write_csv(EVIDENCE_DIR / \'final_per_demo.csv\', demo_rows)\n    write_csv(EVIDENCE_DIR / \'conditioning_sensitivity.csv\', sensitivity_rows)\n    if \'best\' in summary[\'checkpoints\'] and \'last\' in summary[\'checkpoints\']:\n        best_details, last_details = summary[\'checkpoints\'][\'best\'], summary[\'checkpoints\'][\'last\']\n        if best_details.get(\'status\') == last_details.get(\'status\') == \'COMPLETED\':\n            best_d = {r[\'demo_id\']:r[\'ema_loss\'] for r in demo_rows if r[\'checkpoint\']==\'best\'}\n            last_d = {r[\'demo_id\']:r[\'ema_loss\'] for r in demo_rows if r[\'checkpoint\']==\'last\'}\n            bp = [np.mean([r[\'loss\'] for r in profiles if r[\'checkpoint\']==\'best\' and r[\'timestep\']==t]) for t in range(100)]\n            lp = [np.mean([r[\'loss\'] for r in profiles if r[\'checkpoint\']==\'last\' and r[\'timestep\']==t]) for t in range(100)]\n            summary[\'best_to_last\'] = {\n                \'validation_increase_pct\': 100*(last_details[\'ema_demo_mean\']/best_details[\'ema_demo_mean\']-1),\n                \'demos_worsened\': sum(last_d[d]>best_d[d] for d in best_d), \'total_demos\': len(best_d),\n                \'profile_timesteps_worsened\': sum(b>a for a,b in zip(bp,lp)),\n                \'profile_scope\': f\'{len(PROFILE_INDICES)} fixed samples, 1 noise draw, all 100 timesteps; descriptive\'}\n    if sensitivity_rows:\n        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))\n        modes = [\'vision_swapped\',\'proprio_swapped\',\'both_swapped\']\n        for ax,label in zip(axes,[\'best\',\'last\']):\n            rows = [r for r in sensitivity_rows if r[\'checkpoint\']==label]\n            for mode in modes:\n                values = []\n                for t in CFG[\'condition_probe_timesteps\']:\n                    base = [r[\'loss\'] for r in rows if r[\'mode\']==\'matched\' and r[\'timestep\']==t]\n                    alt = [r[\'loss\'] for r in rows if r[\'mode\']==mode and r[\'timestep\']==t]\n                    values.append(float(np.mean(alt)-np.mean(base)) if base else float(\'nan\'))\n                ax.plot(CFG[\'condition_probe_timesteps\'], values, marker=\'.\', label=mode)\n            ax.axhline(0,color=\'gray\',lw=.7)\n            ax.set(title=label+\' EMA: swapped minus matched\',xlabel=\'diffusion timestep\',ylabel=\'delta epsilon MSE\')\n            ax.legend(fontsize=7); ax.grid(alpha=.2)\n        fig.suptitle(\'Counterfactual sensitivity; not closed-loop success\',fontsize=11)\n        fig.tight_layout(); fig.savefig(EVIDENCE_DIR / \'conditioning_sensitivity.png\',dpi=180)\n        fig.savefig(EVIDENCE_DIR / \'conditioning_sensitivity.pdf\'); plt.close(fig)\n    if demo_rows:\n        fig, axes = plt.subplots(1, 2, figsize=(18, 6))\n        for label in sorted({r[\'checkpoint\'] for r in demo_rows}):\n            rows = [r for r in demo_rows if r[\'checkpoint\'] == label]\n            axes[0].plot([r[\'demo_id\'] for r in rows], [r[\'ema_loss\'] for r in rows], marker=\'o\', label=label+\' EMA\')\n            profile = [r for r in profiles if r[\'checkpoint\'] == label]\n            values = [np.mean([r[\'loss\'] for r in profile if r[\'timestep\']==t]) for t in range(100)]\n            axes[1].plot(range(100), values, label=label+\' EMA\')\n        axes[0].tick_params(axis=\'x\', rotation=55)\n        axes[0].set(title=\'Full validation / equal fixed draws\', ylabel=\'epsilon MSE\')\n        axes[1].set(title=\'100 timesteps / fixed 2 samples per demo\', xlabel=\'diffusion timestep\', ylabel=\'epsilon MSE\')\n        for ax in axes:\n            ax.grid(alpha=.2); ax.legend()\n        fig.tight_layout(); fig.savefig(EVIDENCE_DIR / \'diagnostics.png\', dpi=160); plt.close(fig)\n    summary[\'status\'] = (\'COMPLETED\' if all(v.get(\'status\') in [\'COMPLETED\',\'SAME_STEP_AS_BEST\']\n        for v in summary[\'checkpoints\'].values()) else \'PARTIAL\' if seen_steps else \'NOT_AVAILABLE_NO_CHECKPOINT\')\n    atomic_json(EVIDENCE_DIR / \'final_diagnostics.json\', summary)\n    return summary\n\ntry:\n    DIAGNOSTICS = final_diagnostics()\nexcept Exception as exc:\n    # Diagnostic failure must not erase completed training or prevent the final provenance summary.\n    DIAGNOSTICS = {\'run_id\': RUN_ID, \'status\': \'FAILED\', \'error\': str(exc), \'traceback\': traceback.format_exc()}\n    atomic_json(EVIDENCE_DIR / \'diagnostic_failure.json\', DIAGNOSTICS)\n    print(\'Diagnostics failed; checkpoints/core CSV/graphs remain saved:\', repr(exc))\ngc.collect(); torch.cuda.empty_cache()\n\n', '11_sampling.py': "# Paired sampling diagnostics only: weights stay fixed, no optimizer and no simulator.\n# Clipping here is an explicit inference intervention, not the training objective.\nRAW_ACTION_LO = np.concatenate([CACHE[d]['action_targets'].reshape(-1,7) for d in train_ids]).min(0).astype(np.float32)\nRAW_ACTION_HI = np.concatenate([CACHE[d]['action_targets'].reshape(-1,7) for d in train_ids]).max(0).astype(np.float32)\nZ_ACTION_LO = torch.tensor((RAW_ACTION_LO-act_mean)/act_std,device=DEVICE)\nZ_ACTION_HI = torch.tensor((RAW_ACTION_HI-act_mean)/act_std,device=DEVICE)\n\n@torch.inference_mode()\ndef paired_sampling(model,batch,seed):\n    model.eval()\n    condition = model.encode_observation(batch).float()\n    generator_a = torch.Generator(device=DEVICE).manual_seed(seed)\n    generator_b = torch.Generator(device=DEVICE).manual_seed(seed)\n    shape = (len(batch['actions']),16,7)\n    unbounded = torch.randn(shape,device=DEVICE,generator=generator_a)\n    bounded = torch.randn(shape,device=DEVICE,generator=generator_b)\n    assert torch.equal(unbounded,bounded)\n    traces = []\n    for t in range(CFG['diffusion_steps']-1,-1,-1):\n        ts = torch.full((len(bounded),),t,device=DEVICE,dtype=torch.long)\n        eps_unbounded = model.unet(unbounded,condition,ts).float()\n        eps_bounded = model.unet(bounded,condition,ts).float()\n        x0 = (bounded-(1-DIFFUSION.abar[t]).sqrt()*eps_bounded)/DIFFUSION.abar[t].sqrt()\n        clipped = torch.minimum(torch.maximum(x0,Z_ACTION_LO),Z_ACTION_HI)\n        mean = DIFFUSION.betas[t]*DIFFUSION.prev[t].sqrt()/(1-DIFFUSION.abar[t])*clipped\n        mean = mean+(1-DIFFUSION.prev[t])*DIFFUSION.alphas[t].sqrt()/(1-DIFFUSION.abar[t])*bounded\n        if t > 0:\n            noise = torch.randn(shape,device=DEVICE,generator=generator_b)\n            bounded = mean+DIFFUSION.variance[t].sqrt()*noise\n        else:bounded = clipped  # Exact t=0 clean posterior, as in the RGB pilot.\n        unbounded = DIFFUSION.step(eps_unbounded,t,unbounded,generator_a)\n        flags = torch.stack([torch.isfinite(bounded).all(),torch.isfinite(unbounded).all()])\n        assert bool(flags.all()), f'Non-finite sampled trajectory at diffusion timestep {t}'\n        projection_fraction = ((x0 < Z_ACTION_LO)|(x0 > Z_ACTION_HI)).float().mean()\n        rms = bounded.square().mean().sqrt()\n        traces.append({'timestep':t,'projection_fraction':float(projection_fraction),'bounded_latent_rms':float(rms)})\n    std = torch.tensor(act_std,device=DEVICE);mean = torch.tensor(act_mean,device=DEVICE)\n    return {'unbounded':(unbounded*std+mean).cpu().numpy(),'train_range_x0':(bounded*std+mean).cpu().numpy()},traces\n\ndef run_sampling_diagnostics():\n    path,saved = load_latest_verified('best')\n    assert saved is not None, 'No trained best checkpoint is available.'\n    model = VisualConditionedDiffusionPolicy().to(DEVICE).eval()\n    model.load_state_dict(saved['ema'],strict=True);del saved\n    before = model_fingerprint(model)\n    indices = PROFILE_INDICES[:EVAL_OPTIONS['sampling_windows']]\n    assert indices and len(EVAL_OPTIONS['sampling_seeds']) >= 2, 'Diversity requires at least two sampling seeds.'\n    batch = batch_from(val_dataset,indices,'center')\n    outputs = {'unbounded':[],'train_range_x0':[]};trace_rows = []\n    with isolated_rng():\n        for seed in EVAL_OPTIONS['sampling_seeds']:\n            values,trace = paired_sampling(model,batch,seed)\n            for mode,value in values.items():outputs[mode].append(value)\n            trace_rows.extend({'run_id':RUN_ID,'seed':seed,**row} for row in trace)\n    assert model_fingerprint(model) == before, 'Sampling mutated model weights.'\n    metrics = {}\n    for mode,draws in outputs.items():\n        value = np.stack(draws)\n        outside = (value < RAW_ACTION_LO-1e-5)|(value > RAW_ACTION_HI+1e-5)\n        boundary = np.isclose(value,RAW_ACTION_LO,atol=1e-5,rtol=0)|np.isclose(value,RAW_ACTION_HI,atol=1e-5,rtol=0)\n        metrics[mode] = {'max_abs_raw_action':float(np.abs(value).max()),'fraction_outside_train_range':float(outside.mean()),\n            'boundary_fraction':float(boundary.mean()),'boundary_fraction_per_action':boundary.mean((0,1,2)).tolist(),\n            'seed_diversity_pairwise_raw_rms':float(np.sqrt(np.mean((value[0]-value[1])**2))),\n            'normalized_action_rms':float(np.sqrt(np.mean(((value-act_mean)/act_std)**2)))}\n        if mode == 'train_range_x0':assert not outside.any(), 'Bounded sampler failed its range contract.'\n    with open(EVIDENCE_DIR/'sampling_outputs.npz','wb') as f:\n        np.savez_compressed(f,**{k:np.stack(v) for k,v in outputs.items()},raw_lo=RAW_ACTION_LO,raw_hi=RAW_ACTION_HI,\n                            sample_indices=np.asarray(indices),seeds=np.asarray(EVAL_OPTIONS['sampling_seeds']))\n    write_csv(EVIDENCE_DIR/'sampling_projection_trace.csv',trace_rows)\n    report = {'status':'COMPLETED','run_id':RUN_ID,'checkpoint':str(path),'checkpoint_sha256':sha256_file(path),\n        'model':'EMA FP32','windows':len(indices),'seeds':EVAL_OPTIONS['sampling_seeds'],'metrics':metrics,\n        'paired_noise':True,'bounds_scope':'empirical training targets only, transformed to saved z-score coordinates',\n        'limitations':'Range compliance is expected by construction, not action accuracy, controller certification or Lift success. No rollout is performed.'}\n    atomic_json(EVIDENCE_DIR/'sampling_diagnostics.json',report)\n    fig,axes = plt.subplots(1,2,figsize=(11,4))\n    for seed in EVAL_OPTIONS['sampling_seeds']:\n        rows = [r for r in trace_rows if r['seed'] == seed]\n        axes[0].plot([r['timestep'] for r in rows],[r['projection_fraction'] for r in rows],label=f'seed {seed}')\n    axes[0].set(xlabel='diffusion timestep',ylabel='projected fraction',title='Bounded x0 projection',ylim=(0,1));axes[0].invert_xaxis();axes[0].legend()\n    axes[1].bar(list(metrics),[metrics[k]['fraction_outside_train_range'] for k in metrics])\n    axes[1].set(ylabel='fraction outside empirical training bounds',title='Sampling diagnostic only',ylim=(0,1))\n    for ax in axes:ax.grid(axis='y',alpha=.2)\n    fig.tight_layout();fig.savefig(EVIDENCE_DIR/'sampling_diagnostics.png',dpi=150);plt.close(fig)\n    del model;gc.collect();torch.cuda.empty_cache()\n    return report\n\ntry:\n    SAMPLING = run_sampling_diagnostics()\nexcept Exception as exc:\n    SAMPLING = {'status':'FAILED','error':str(exc),'traceback':traceback.format_exc(),'run_id':RUN_ID}\n    atomic_json(EVIDENCE_DIR/'sampling_failure.json',SAMPLING)\n    print('Sampling diagnostics failed; training evidence and checkpoints remain available:',repr(exc))\nprint('Sampling status:',SAMPLING['status'])\n", '12_summary.py': "export_efficiency(TRAINING_STATE)\ndurable=json.loads((RUN_DIR/'status.json').read_text()) if (RUN_DIR/'status.json').exists() else {'durable_step':0}\nbest=DIAGNOSTICS.get('checkpoints',{}).get('best',{})\nbest_metric=best.get('ema_demo_mean',TRAINING_STATE['best_metric'])\nreference_best=min(RGB_HISTORY,key=lambda r:r['ema_demo_mean']) if RGB_HISTORY else None\nSUMMARY={'run_id':RUN_ID,'session_id':SESSION_ID,'mode':MODE,'utc':utc_now(),'modality':MODALITY,'dataset_id':EXPECTED_DATASET_ID,\n    'status':TRAINING_STATE['status'],'stop_reason':TRAINING_STATE['stop_reason'],\n    'pipeline_status':'COMPLETED' if TRAINING_STATE['status']=='COMPLETED' and DIAGNOSTICS['status']=='COMPLETED' and SAMPLING['status']=='COMPLETED' else 'PARTIAL',\n    'started_utc':TRAINING_STATE['started_utc'],'ended_utc':TRAINING_STATE['ended_utc'],\n    'observed_step':TRAINING_STATE['step'],'durable_step':durable['durable_step'],\n    'best_step':best.get('step',TRAINING_STATE['best_step']),'best_metric':best_metric,\n    'selection_metric':'full40_validation_ema_equal_demo_mean_fixed_3_draws',\n    'training_budget_updates':CFG['max_steps'],'lr_schedule_horizon':CFG['lr_schedule_steps'],\n    'denoising_early_stopping_enabled':CFG['early_stop_enabled'],\n    'common_validation_demos':COMMON_VAL_IDS,\n    'best_common_validation':best.get('ema_common_demo_mean'),\n    'final_common_validation':TRAINING_STATE['history'][-1].get('ema_common_demo_mean') if TRAINING_STATE['history'] else None,'training_seed':CFG['seed'],'split_seed':CFG['split_seed'],\n    'config_hash':CONFIG_HASH,'data_fingerprint':DATA_FINGERPRINT,'normalization_hash':NORMALIZATION_HASH,\n    'code_version':CODE_VERSION,'code_sha256':CODE_SHA256,'training_runtime':TRAINING_STATE['training_runtime'],'report_runtime':RUNTIME,\n    'parameters_total':PREFLIGHT['parameters_total'],'parameters_trainable':PREFLIGHT['parameters_trainable'],\n    'tail_target_policy':CFG['tail_target_policy'],'export_schema':3,\n    'training_samples':len(train_dataset),'validation_samples':len(val_dataset),\n    'equivalent_data_passes':TRAINING_STATE['step']*64/len(train_dataset),\n    'preflight_status':PREFLIGHT['status'],'diagnostics_status':DIAGNOSTICS['status'],'sampling_status':SAMPLING['status'],\n    'new_rgb_reference_run_id':REFERENCE_RUN_ID if reference_best else None,'new_rgb_reference_best':reference_best,\n    'relative_best_denoising_change_pct':None if reference_best is None or best_metric is None else 100*(best_metric/reference_best['ema_demo_mean']-1),\n    'closed_loop_evaluated':False,'final_test_performed':False,'outputs_root':str(RUN_DIR),\n    'session_wall_seconds':time.perf_counter()-NOTEBOOK_START_PERF,\n    'next_step':'Review both NEW matched training runs; update Notebook07 to these exact dataset/checkpoint/normalization identities and the validated NVIDIA source before development rollouts.'}\natomic_json(EVIDENCE_DIR/'run_summary.json',SUMMARY);atomic_json(SESSION_DIR/'run_summary.json',SUMMARY)\nreport=f'''# Notebook {'04' if MODALITY=='rgb' else '05'} — NEW paired NVIDIA D200 TAIL execution\n\n- Modality: {MODALITY}; Run ID: `{RUN_ID}`; session: `{SESSION_ID}`; mode: `{MODE}`.\n- Training: **{SUMMARY['status']}** / `{SUMMARY['stop_reason']}`; whole pipeline: {SUMMARY['pipeline_status']}.\n- Fixed budget / LR schedule horizon: {CFG['max_steps']} / {CFG['lr_schedule_steps']}; denoising early stopping: {CFG['early_stop_enabled']}.\n- Common10 best/final EMA validation: {SUMMARY['best_common_validation']} / {SUMMARY['final_common_validation']}.\n- Optimizer updates observed/durable: {SUMMARY['observed_step']}/{SUMMARY['durable_step']}.\n- Best EMA equal-demo denoising loss: {best_metric} at {SUMMARY['best_step']}.\n- Diagnostics: {DIAGNOSTICS['status']}; sampling: {SAMPLING['status']}.\n- Total/trainable parameters: {SUMMARY['parameters_total']:,}/{SUMMARY['parameters_trainable']:,}.\n- Effective batch: {CFG['micro_batch']} x {CFG['accumulation']} = 64. Equivalent overlapping-window passes: {SUMMARY['equivalent_data_passes']:.2f}.\n- Dataset: `{EXPECTED_DATASET_ID}`; samples: {len(train_dataset)} train / {len(val_dataset)} validation.\n- Paired data fingerprint: `{DATA_FINGERPRINT}`; normalization hash: `{NORMALIZATION_HASH}`.\n- Config hash: `{CONFIG_HASH}`; shared code build: `{CODE_SHA256}`.\n- New paired RGB reference: {reference_best}; relative denoising change: {SUMMARY['relative_best_denoising_change_pct']}%.\n- Artifacts: `{RUN_DIR}`.\n\nBoth modalities use the same schema3 tail-complete rows, action targets, robot-only36D state, fixed validation noise, architecture and training/selection protocol. RGB uses fixed ImageNet constants; Events use saved training-only per-camera channel z-score without another log transform or division by255. No preprocessing statistics are refitted here. Endpoint-repeated targets participate in ordinary unmasked epsilon MSE; action_is_recorded is provenance, not a loss mask. Validation includes descriptive fully-recorded versus endpoint-repeated input strata without changing the primary metric. Historical schema2 denoising metrics cover a different population and normalization and are not directly comparable to these schema3 losses. Historical CPU runs are separate experiments.\n\nDisposable CUDA preflight is followed by a seeded production rebuild. Only this same-ID verified resume may load optimizer/model state; ImageNet is the only pretrained encoder initialization. Two verified full resume generations survive publication; all strict best improvements and every500-step EMA milestone remain. A completed/failed terminal run and evaluate mode do not start another optimizer loop. Default training stops at4,500 successful updates with the10,000-step LR schedule retained; denoising early stopping is disabled. Numerical/preflight failures still stop safely. Both best and fixed-budget final checkpoints require behavioral evaluation before deployment selection. Changing code/config/data produces a new identity. Never open concurrent sessions for one Run ID.\n\nGradients are reported before/after clipping, with clip frequency and threshold ratios. Unsafe accumulation blocks are discarded without advancing optimizer/EMA/LR counters. Diagnostics include per-demo losses, all100 diffusion timesteps on80 fixed windows, crop/AMP probes and swapped visual/proprio conditioning. FP32 paired sampling compares unbounded DDPM to empirical train-range x0 projection; range compliance is not action accuracy or Lift success.\n\nSynthetic Events come from interpolated simulator visual states, not physical Event-camera measurements. Source preparation passed; D200 task performance remains unknown until matched development rollouts. D50 denoising degradation did not predict the observed late RGB rollout improvement. D200 is development data used for selection. Descriptive forty-demo bootstrap intervals do not include model-selection, training-seed or rollout uncertainty. No closed-loop episode or final evaluation was performed here. The old Notebook07 checkpoint/source configuration is not automatically updated.\n\n{SUMMARY['next_step']}\n'''\nfor folder in [EVIDENCE_DIR,SESSION_DIR]: (folder/'run_summary.md').write_text(report,encoding='utf-8')\nhandoff=f'''Actual NEW paired NVIDIA D200 TAIL training update. Use the attached evidence, preserve historical experiment records.\nModality={MODALITY}; Run ID={RUN_ID}; session={SESSION_ID}; mode={MODE}.\nTraining={SUMMARY['status']}/{SUMMARY['stop_reason']}; pipeline={SUMMARY['pipeline_status']}; best={best_metric}@{SUMMARY['best_step']}.\nObserved/durable updates={SUMMARY['observed_step']}/{SUMMARY['durable_step']}.\nDataset={EXPECTED_DATASET_ID}; paired fingerprint={DATA_FINGERPRINT}; normalization={NORMALIZATION_HASH}; config={CONFIG_HASH}; code={CODE_SHA256}.\nDiagnostics={DIAGNOSTICS['status']}; sampling={SAMPLING['status']}; output={RUN_DIR}.\nNo policy rollout or final test. Do not infer Lift success from denoising or bounded sampling. Do not compare historical CPU runs as if they used the new NVIDIA source.\nUpdate PROJECT_STATUS/EXPERIMENT_LOG/DECISIONS/DATA_DICTIONARY only from actual evidence. Preserve split20260927, training42/43/44, final evaluation1000–1049. No assumed modality winner. Future Notebook07 must use exact new checkpoints, normalization and the accepted source protocol.\n'''\nprompt_name='PROMPT_Notebook04_EN.md' if MODALITY=='rgb' else 'PROMPT_Notebook05_EN.md'\nfor name in ['PROMPT_Project_Control_EN.md',prompt_name]: (EVIDENCE_DIR/name).write_text(handoff,encoding='utf-8')\nfor name in ['final_diagnostics.json','diagnostic_failure.json','sampling_diagnostics.json','sampling_failure.json','PROMPT_Project_Control_EN.md',prompt_name]:\n    path=EVIDENCE_DIR/name\n    if path.exists(): shutil.copyfile(path,SESSION_DIR/name)\nmanifest=[]\nfor path in sorted(EVIDENCE_DIR.rglob('*')):\n    if path.is_file() and path.name not in ['artifact_manifest.csv','evidence_bundle.zip'] and '.tmp' not in path.name:\n        manifest.append({'run_id':RUN_ID,'relative_path':str(path.relative_to(EVIDENCE_DIR)),'bytes':path.stat().st_size,'sha256':sha256_file(path)})\nwrite_csv(EVIDENCE_DIR/'artifact_manifest.csv',manifest)\nimport zipfile\nbundle=RUN_DIR/'evidence_bundle.zip'\nwith zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:\n    for path in sorted(EVIDENCE_DIR.rglob('*')):\n        if path.is_file() and '.tmp' not in path.name: z.write(path,str(path.relative_to(EVIDENCE_DIR)))\nprint(report)\nfrom IPython.display import display,Image\nfor name in ['visual_input_preview.png','training_dashboard.png','validation_full_common.png','rgb_events_comparison.png','efficiency_dashboard.png','diagnostics.png','conditioning_sensitivity.png','sampling_diagnostics.png']:\n    path=EVIDENCE_DIR/name\n    if path.exists(): display(Image(filename=str(path)))\nprint('Share this evidence bundle:',bundle)\n"}
ORCHESTRATOR_SOURCE='"""Run the unchanged, verified seed42 scientific engine for seeds43/44 only."""\nimport copy, csv, gc, hashlib, json, os, time, traceback, uuid, zipfile\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nORCHESTRATOR_VERSION = \'notebook04-05-tail-paired-seeds43-44-v1\'\n\ndef canonical_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\ndef atomic_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp-\' + uuid.uuid4().hex)\n    try:\n        tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n        os.replace(tmp, path)\n    finally:\n        tmp.unlink(missing_ok=True)\n\ndef child(root, relative):\n    root = Path(root).resolve(); path = (root / relative).resolve()\n    assert path.is_relative_to(root), \'Artifact path escaped its run directory.\'\n    return path\n\ndef identity(cfg, meta):\n    digest = canonical_hash({\'code\': meta[\'code_version\'], \'code_sha256\': meta[\'shared_build_sha256\'],\n        \'config\': cfg, \'data\': meta[\'data_fingerprint\'], \'normalization\': meta[\'normalization_hash\']})\n    run_id = f"d200_{cfg[\'modality\']}_paired_gpu_tail_fixed4500_v1_seed{cfg[\'seed\']}_{digest[:12]}"\n    return run_id, digest\n\ndef expected_record(cfg, meta):\n    run_id, digest = identity(cfg, meta)\n    rgb = copy.deepcopy(cfg); rgb.update(modality=\'rgb\', visual_normalization=\'fixed_imagenet\')\n    return {\'run_id\': run_id, \'config_hash\': digest, \'code_version\': meta[\'code_version\'],\n        \'code_sha256\': meta[\'shared_build_sha256\'], \'config\': cfg,\n        \'data_fingerprint\': meta[\'data_fingerprint\'], \'paired_sample_fingerprint\': meta[\'data_fingerprint\'],\n        \'normalization_hash\': meta[\'normalization_hash\'], \'dataset_id\': meta[\'dataset_id\'],\n        \'modality\': cfg[\'modality\'], \'rgb_reference_run_id\': identity(rgb, meta)[0] if cfg[\'modality\']==\'events\' else None}\n\ndef completed_run(root, cfg, meta):\n    """Skip only fully published, checksum-verified completion; never infer success from a folder."""\n    expected = expected_record(cfg, meta); directory = Path(root)/\'artifacts/checkpoints\'/cfg[\'modality\']/expected[\'run_id\']\n    record_path = directory/\'run_config.json\'\n    if not record_path.exists():\n        assert not directory.exists() or not any(directory.rglob(\'*.ckpt\')), \'Orphan checkpoints block a fresh run.\'\n        return None\n    assert json.loads(record_path.read_text()) == expected, \'Existing run identity differs.\'\n    assert canonical_hash(json.loads((directory/\'normalization.json\').read_text())) == meta[\'normalization_hash\']\n    status_path = directory/\'status.json\'\n    status = json.loads(status_path.read_text()) if status_path.exists() else {}\n    if status.get(\'durable_step\', 0)>0:\n        assert list((directory/\'resume\').glob(\'step_*.ckpt\')), \'Durable training has no resume generation; refusing a fresh restart.\'\n    summary_path = directory/\'evidence/run_summary.json\'\n    if not summary_path.exists(): return None\n    summary = json.loads(summary_path.read_text())\n    if summary.get(\'pipeline_status\') != \'COMPLETED\': return None\n    for key, value in [(\'run_id\', expected[\'run_id\']), (\'config_hash\', expected[\'config_hash\']),\n                       (\'data_fingerprint\', meta[\'data_fingerprint\']), (\'normalization_hash\', meta[\'normalization_hash\']),\n                       (\'training_seed\', cfg[\'seed\']), (\'code_sha256\', meta[\'shared_build_sha256\']),\n                       (\'durable_step\', cfg[\'max_steps\']), (\'observed_step\', cfg[\'max_steps\']),\n                       (\'stop_reason\', \'budget_exhausted\'), (\'status\', \'COMPLETED\')]:\n        assert summary.get(key)==value, f\'Completed summary mismatch: {key}\'\n    assert status[\'status\']==\'COMPLETED\' and status[\'durable_step\']==cfg[\'max_steps\'] and status[\'stop_reason\']==\'budget_exhausted\'\n    for kind in [\'best\', \'last\']:\n        pointer = json.loads((directory/(kind+\'.json\')).read_text())\n        assert pointer[\'run_id\']==expected[\'run_id\'] and pointer[\'config_hash\']==expected[\'config_hash\']\n        assert 0<int(pointer[\'step\'])<=cfg[\'max_steps\']\n        if kind==\'last\': assert pointer[\'step\']==cfg[\'max_steps\']\n        path = child(directory, pointer[\'relative_path\'])\n        assert path.parent.name==(\'best\' if kind==\'best\' else \'resume\')\n        assert file_hash(path)==pointer[\'sha256\'], \'Published completed checkpoint checksum failed.\'\n        sidecar = json.loads(path.with_suffix(path.suffix+\'.sha256.json\').read_text())\n        assert sidecar[\'sha256\']==pointer[\'sha256\'] and sidecar[\'bytes\']==path.stat().st_size\n    with open(directory/\'evidence/artifact_manifest.csv\', newline=\'\') as f:\n        rows = list(csv.DictReader(f))\n    assert rows, \'Missing evidence manifest.\'\n    assert any(row[\'relative_path\']==\'run_summary.json\' for row in rows)\n    for row in rows:\n        path = child(directory/\'evidence\', row[\'relative_path\'])\n        assert row[\'run_id\']==expected[\'run_id\'] and path.stat().st_size==int(row[\'bytes\'])\n        assert file_hash(path)==row[\'sha256\'], \'Completed evidence checksum failed.\'\n    bundle = directory/\'evidence_bundle.zip\'\n    with zipfile.ZipFile(bundle) as z:\n        assert z.testzip() is None\n        for row in rows:\n            assert hashlib.sha256(z.read(row[\'relative_path\'])).hexdigest()==row[\'sha256\'], \'Evidence bundle differs from its manifest.\'\n    return summary\n\ndef release_namespace(ns):\n    torch = ns.get(\'torch\')\n    ns.clear(); gc.collect()\n    if torch is not None and torch.cuda.is_available(): torch.cuda.empty_cache()\n\ndef execute_seed(seed, mode, modality, modules, meta, cfg, root, queue_id, orchestrator_sha):\n    ns = {\'__name__\':\'__main__\', \'MODALITY\':modality}\n    try:\n        for name in sorted(modules):\n            print(f\'=== seed {seed} | {modality} | {name} ===\', flush=True)\n            exec(compile(modules[name].replace(\'__BUILD_CODE_SHA256__\', meta[\'shared_build_sha256\']), name, \'exec\'), ns)\n            if name==\'01_setup.py\':\n                expected = copy.deepcopy(ns[\'CFG\']); expected[\'seed\']=seed\n                assert expected==cfg, \'Queue recipe differs from the frozen engine configuration.\'\n                ns[\'CFG\'][\'seed\']=seed; ns[\'MODE\']=mode\n            if name==\'02_data.py\':\n                assert ns[\'RUN_ID\']==identity(cfg,meta)[0]\n                atomic_json(ns[\'EVIDENCE_DIR\']/\'replica_orchestration.json\', {\n                    \'version\':ORCHESTRATOR_VERSION, \'orchestrator_sha256\':orchestrator_sha,\n                    \'queue_id\':queue_id, \'seed\':seed, \'mode\':mode,\n                    \'scientific_engine_sha256\':meta[\'shared_build_sha256\'],\n                    \'configuration_override\':{\'seed\':seed}, \'fresh_namespace\':True,\n                    \'seed42_checkpoint_loaded\':False,\n                    \'scope\':\'Sequential execution wrapper; scientific source modules unchanged.\'})\n            if name==\'06_checkpoints.py\':\n                snapshot=\'\\n\\n\'.join(\'# SOURCE MODULE: \'+key+\'\\n\'+modules[key] for key in sorted(modules)).replace(\'__BUILD_CODE_SHA256__\',meta[\'shared_build_sha256\'])\n                assert hashlib.sha256(snapshot.encode()).hexdigest()==meta[\'source_snapshot_sha256\']\n                (ns[\'SESSION_DIR\']/\'scientific_source_snapshot.py\').write_text(snapshot)\n            if name==\'09_train.py\':\n                state=ns[\'TRAINING_STATE\']\n                assert state[\'status\']==\'COMPLETED\' and state[\'step\']==cfg[\'max_steps\'], f"Training stopped: {state[\'status\']} / {state[\'stop_reason\']}. Queue paused; rerun uses durable same-ID state."\n        summary=copy.deepcopy(ns[\'SUMMARY\'])\n        assert summary[\'pipeline_status\']==\'COMPLETED\', \'Training finished but diagnostics/sampling did not; queue paused.\'\n        return summary\n    finally:\n        release_namespace(ns)\n\ndef run_queue(modality, seeds, mode, modules, meta, recipes, root, orchestrator_sha,\n              runner=execute_seed, checker=completed_run):\n    assert modality in [\'rgb\',\'events\'] and mode in [\'train\',\'evaluate\']\n    assert list(seeds)==[43,44], \'This replication queue is locked to seeds43/44. Seed42 is historical.\'\n    raw=\'\\n\\n\'.join(\'# SOURCE MODULE: \'+name+\'\\n\'+modules[name] for name in sorted(modules))\n    assert hashlib.sha256(raw.encode()).hexdigest()==meta[\'shared_build_sha256\'], \'Scientific source changed.\'\n    assert set(modules)==set(meta[\'module_hashes\'])\n    for name, source in modules.items():\n        assert hashlib.sha256(source.encode()).hexdigest()==meta[\'module_hashes\'][name]\n        compile(source.replace(\'__BUILD_CODE_SHA256__\',meta[\'shared_build_sha256\']),name,\'exec\')\n    protocol={\'orchestrator_version\':ORCHESTRATOR_VERSION,\'orchestrator_sha256\':orchestrator_sha,\n        \'scientific_engine_sha256\':meta[\'shared_build_sha256\'],\'modality\':modality,\n        \'seeds\':list(seeds),\'mode\':mode,\'recipes\':recipes,\'data_fingerprint\':meta[\'data_fingerprint\'],\n        \'normalization_hash\':meta[\'normalization_hash\'],\'closed_loop_evaluated\':False}\n    queue_id=modality+\'_tail_seed43_44_\'+canonical_hash(protocol)[:12]\n    output=Path(root)/\'artifacts/training_replications\'/queue_id\n    session=output/\'sessions\'/(datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')+\'_\'+uuid.uuid4().hex[:8])\n    session.mkdir(parents=True,exist_ok=False); atomic_json(session/\'protocol.json\',protocol)\n    ledger={\'queue_id\':queue_id,\'status\':\'RUNNING\',\'session\':str(session),\'protocol\':protocol,\'runs\':[],\n        \'started_utc\':datetime.now(timezone.utc).isoformat()}\n    def publish():\n        atomic_json(session/\'queue_summary.json\',ledger);atomic_json(output/\'queue_summary.json\',ledger)\n    try:\n        publish()\n        for seed in seeds:\n            cfg=copy.deepcopy(recipes[str(seed)])\n            assert cfg[\'seed\']==seed and cfg[\'modality\']==modality\n            ledger[\'active_run\']={\'seed\':seed,\'run_id\':identity(cfg,meta)[0],\n                \'directory\':str(Path(root)/\'artifacts/checkpoints\'/modality/identity(cfg,meta)[0])}\n            publish()\n            # Require the exact paired RGB completion before any Event training.\n            if modality==\'events\':\n                rgb=copy.deepcopy(cfg);rgb.update(modality=\'rgb\',visual_normalization=\'fixed_imagenet\')\n                assert checker(root,rgb,meta) is not None, f\'Complete Notebook04 RGB seed {seed} first.\'\n            summary=checker(root,cfg,meta)\n            reused=summary is not None\n            if reused: print(\'Skipping verified completed run:\',summary[\'run_id\'],flush=True)\n            else: summary=runner(seed,mode,modality,modules,meta,cfg,root,queue_id,orchestrator_sha)\n            # Read-only integrity audit after newly produced reports; no second optimizer loop.\n            if not reused:\n                verified=checker(root,cfg,meta)\n                assert verified is not None and verified[\'config_hash\']==summary[\'config_hash\']\n            run_dir=Path(root)/\'artifacts/checkpoints\'/modality/summary[\'run_id\']\n            ledger[\'runs\'].append({\'seed\':seed,\'reused_verified_completion\':reused,\'summary\':summary,\n                \'evidence_bundle\':str(run_dir/\'evidence_bundle.zip\'),\n                \'evidence_bundle_sha256\':file_hash(run_dir/\'evidence_bundle.zip\')})\n            publish()\n        ledger[\'status\']=\'COMPLETED\'\n        ledger[\'active_run\']=None\n    except BaseException as exc:\n        ledger[\'status\']=\'ABORTED\' if isinstance(exc,KeyboardInterrupt) else \'FAILED\'\n        ledger[\'error\']=str(exc);ledger[\'traceback\']=traceback.format_exc()\n        raise\n    finally:\n        ledger[\'ended_utc\']=datetime.now(timezone.utc).isoformat();publish()\n        bundle=output/\'replication_evidence_bundle.zip\'\n        temp=bundle.with_name(bundle.name+\'.tmp-\'+uuid.uuid4().hex)\n        with zipfile.ZipFile(temp,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n            z.write(output/\'queue_summary.json\',\'queue_summary.json\')\n            z.write(session/\'protocol.json\',\'protocol.json\')\n            for row in ledger[\'runs\']:\n                z.write(row[\'evidence_bundle\'],f"seed{row[\'seed\']}/evidence_bundle.zip")\n        os.replace(temp,bundle)\n        print(\'Queue status:\',ledger[\'status\'],\'| Share:\',bundle,flush=True)\n    return ledger\n'
import hashlib
assert hashlib.sha256(ORCHESTRATOR_SOURCE.encode()).hexdigest()==ORCHESTRATOR_SHA256
exec(compile(ORCHESTRATOR_SOURCE,"replica_queue.py","exec"),globals())


## 3. Automatic sequential execution and resumable evidence

Drive authorization may appear once. No manual seed change or cell selection is needed. A core failure or failed diagnostics stops the queue and publishes its status; the next seed is not started. Completed evidence is protected by checkpoint and artifact checksums. Automatic reuse does not load or update model weights.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
PROJECT_ROOT=Path("/content/drive/MyDrive/DiffusionPolicyEvents")
QUEUE_SUMMARY=run_queue(MODALITY,TRAINING_SEEDS,MODE,SOURCE_MODULES,ENGINE_META,RECIPES,PROJECT_ROOT,ORCHESTRATOR_SHA256)


## 4. What to upload and what comes next

Upload the printed `replication_evidence_bundle.zip`; it contains both per-seed evidence bundles and the queue summary, without weight files. Per-run checkpoints, graphs, CSVs, normalization, fingerprints and diagnostics remain in `artifacts/checkpoints/<modality>/<Run ID>/`.

After both notebooks complete, review all three paired training seeds42/43/44. Plan equally budgeted, paired closed-loop development evaluations with the same sampler/source/reset contract. Retain every seed; do not select a lucky seed or pool episodes as independent trained models. Final evaluation seeds1000–1049 remain locked and unused. No additional Notebook02/03 execution is required for this queue.
